# MGMT Run 4 recovery

This notebook rebuilds the study results from the archived per-model predictions. It runs on CPU and takes the original **Run 4 single-slice model ZIP** or **outputs_merged_run4.zip** as input. It also accepts an unpacked run with an `outputs_merged/jobs` directory.

1. In Kaggle, import this notebook and attach the Run 4 archive using **Add Input**.
2. Run all cells. If multiple Run 4 archives are attached, set `RUN4_ARCHIVE` to the one you want in the configuration cell.
3. Download `MGMT_Run4_rebuilt_results.zip` from the notebook outputs.

The original v9 training, input-checking, and plotting scripts are included in the source package. The results below are rebuilt from archived predictions. The archive does not contain the original executed Kaggle notebook or a training-source checksum.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

# Enter a full path here to select a specific archive.
RUN4_ARCHIVE = os.environ.get("MGMT_RUN4_ARCHIVE", "")
WORK_DIR = Path(os.environ.get(
    "MGMT_RUN4_WORK_DIR",
    "/kaggle/working" if Path("/kaggle/working").is_dir() else "./mgmt_run4_work",
)).expanduser().resolve()
SOURCE_DIR = WORK_DIR / "mgmt_run4_source"
OUTPUT_DIR = WORK_DIR / "mgmt_run4_rebuilt_results"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
RECOVERED_FILES = {
    'kaggle_merged_training.py': r'''#!/usr/bin/env python3
# =============================================================================
# kaggle_merged_training.py
# -----------------------------------------------------------------------------
# Multi-cohort training of the proposed tumour-focused 2.5D EfficientNet, using
# BOTH T4 GPUs at the same time.
#
# WHAT IT DOES
#   Design 1 (main result, like-for-like with the paper):
#     The 574 matched RSNA-MICCAI + BraTS 2021 patients keep EXACTLY the same
#     5-fold splits as the paper (StratifiedKFold, seeds 42 and 202). In every
#     fold, the training set is  BraTS train-fold  +  ALL UPenn-GBM / UCSF-PDGM
#     patients.  The held-out BraTS fold is scored as before, so the new AUC is
#     measured on the SAME 574 patients as the 0.669 headline and can be compared
#     with a DeLong test.
#   Control (RUN_CONTROL): EfficientNet-B0 on BraTS only, same code. Isolates the
#     effect of the extra data from any code differences.
#   Design 2 (RUN_EXTERNAL): leave-one-site-out. Train on BraTS + the other extra
#     site, test on the held-out site that no model has seen.
#
# HOW BOTH GPUs ARE USED
#   Every (backbone x modality x seed x fold) model is an independent job. Two
#   worker processes, one pinned to each T4, pull jobs from a shared queue, so
#   both GPUs train different models simultaneously (~2x throughput). For small
#   backbones like EfficientNet-B0/B2 this is faster and more stable than
#   splitting one model across GPUs (DataParallel/DDP).
#   Watch `!nvidia-smi` in another cell: both GPUs should show ~90%+ use.
#
# LEAK PROTECTION (UPenn and UCSF patients also appear in BraTS 2021)
#   * UCSF-PDGM metadata carries a BraTS21 ID column -> exact matches removed.
#   * Image-similarity audit of every extra patient vs the 574 evaluation
#     patients; anything above DEDUP_THRESHOLD is removed from training AND from
#     external testing. A false positive only costs one training case, so the
#     threshold is deliberately conservative. See dedup_report.csv.
#
# KAGGLE SETUP
#   Accelerator: GPU T4 x2.  Internet: ON (for ImageNet weights).
#   Inputs (Add Input):
#     1. RSNA-MICCAI Brain Tumor Radiogenomic Classification (competition)
#     2. BraTS 2021 Task 1 dataset (tar or extracted folders)
#     3. UPenn-GBM subset  -> <SID>_{FLAIR,T1,T1GD,T2,SEG/segm}.nii.gz
#                             + upenn_mgmt_labels.csv  (from upenn_1_fetch_labels.py)
#     4. UCSF-PDGM subset  -> UCSF-PDGM-xxxx_{FLAIR,T1,T1c,T2,tumor_segmentation}.nii.gz
#                             + UCSF-PDGM-metadata*.csv (from ucsf_subset_on_pc.py)
#     5. (optional) your earlier OOF files: oof_predictions.csv (0.669 headline),
#        oof_improve_*.csv, oof_ablation_nocrop.csv -> used for comparisons and
#        the combined ensemble.
#     6. (optional, to resume) the output of a previous run of this script.
#   Either extra site may be missing; the script uses whatever it finds.
#
# RUN
#   1) Set QUICK_TEST = True, run once (~15 min) to check inputs and GPUs.
#   2) Set QUICK_TEST = False, then "Save Version -> Save & Run All".
#      !python kaggle_merged_training.py
#   If the session stops early, attach this run's output as an input and run
#   again: finished jobs are skipped.
#
# OUTPUT  /kaggle/working/outputs_merged/
#   results_table.csv/.png     main table (AUC, 95% CI, dAUC, DeLong p, MCC, folds)
#   external_table.csv         leave-one-site-out results
#   oof_merged_*.csv           per-patient OOF predictions (for paper figures)
#   cohort_summary.json        patients per site, dedup counts
#   dedup_report.csv           similarity audit details
#   results.json               everything above in one file
#   jobs/*.npz                 per-model results (resume state)
# =============================================================================
import os, sys, re, glob, time, json, random, shutil, tarfile, fnmatch, warnings, traceback
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
os.environ.setdefault("NO_ALBUMENTATIONS_UPDATE", "1")   # no update check on import
try:
    sys.stdout.reconfigure(line_buffering=True)           # show log lines immediately
except Exception:
    pass

# ------------------------------- CONFIG --------------------------------------
SCRIPT_VERSION = "v9 (2026-10-11: single-slice input for every experiment)"
QUICK_TEST  = False      # True -> tiny run (2 epochs, 2 modalities, 1 seed) to check setup

MODALITIES  = ["FLAIR", "T1w", "T1wCE", "T2w"]
SUFFIX      = {"FLAIR": "flair", "T1w": "t1", "T1wCE": "t1ce", "T2w": "t2"}
IMG_SIZE    = 224
N_SLICES    = 3
SLICE_GAP   = 4          # voxels at BraTS 1 mm spacing; rescaled for other spacings
CROP_MARGIN = 15
N_FOLDS     = 5
BATCH       = 32
LR          = 3e-4
WD          = 1e-5
BAD_IDS     = [109, 123, 709]
SEEDS       = [42, 202]  # same seeds = same folds as the paper
EPOCHS      = 20

BACKBONES         = ["efficientnet_b2", "efficientnet_b0"]   # longest first
EXTRA_SITES       = ["UPENN", "UCSF"]
EXTRA_GRADE4_ONLY = True       # UCSF-PDGM has grade 2-4; RSNA/BraTS is grade 4
# UCSF calls MGMT "positive" from index >= 1, but on the 245 UCSF patients that are
# also in RSNA-MICCAI, "index >= 2" reproduces the RSNA label exactly (100%).
# Relabel UCSF with the RSNA definition so both cohorts mean the same thing.
UCSF_MGMT_INDEX_CUTOFF = 2     # None -> use UCSF's own positive/negative status
RUN_CONTROL       = True       # B0 on BraTS only with this exact code
RUN_EXTERNAL      = True       # leave-one-site-out (needs >= 1 extra site)
EXTERNAL_BACKBONE = "efficientnet_b0"
# Standard baselines, trained under the IDENTICAL merged protocol as the proposed
# model (same 574 evaluation patients and folds, same extra cohorts, same 2.5D
# tumour inputs, epochs and augmentation). Report every baseline that is run.
RUN_BASELINES      = True
# Reported baselines = the architectures requested by the publishing author
# (supervisor's instruction, 2026-10-10). VGG-16 and MobileNet-v3-Large were also
# trained in run 2 (jobs base_vgg16_bn_* / base_mobilenetv3_large_100_* in
# outputs_merged_run2.zip) but are not part of the requested comparison set.
BASELINE_BACKBONES = ["densenet121", "resnet18", "mobilenetv2_100", "simple_cnn"]   # longest first
BASELINE_LABELS    = {"simple_cnn": "Simple CNN (4-layer, from scratch)", "vgg16_bn": "VGG-16",
                      "resnet18": "ResNet-18", "densenet121": "DenseNet-121",
                      "mobilenetv2_100": "MobileNet-v2", "mobilenetv3_large_100": "MobileNet-v3-Large"}
# Ablation of the proposed B0+B2 ensemble under the same merged protocol. Each
# variant removes ONE component; "−multi-cohort", "−ensemble", "−skill weighting"
# and "−multimodal fusion" are computed from existing runs without training.
RUN_CONTROL_B2   = True        # BraTS-only B2 (for the "−multi-cohort" ablation row)
RUN_ABLATION     = True
ABLATIONS        = ["center", "nocrop", "single", "randinit"]
ABLATION_LABELS  = {"randinit": "− ImageNet pretraining (random init)",
                    "single":   "− 2.5D context (single slice)",
                    "nocrop":   "− Tumour-focus crop (whole slice, tumour slice)",
                    "center":   "− Tumour guidance (centre slice, whole image)"}
CACHE_VARIANTS   = {"nocrop": dict(use_crop=False, slice_mode="tumor"),
                    "center": dict(use_crop=False, slice_mode="center")}
# Grad-CAM: a few B2 models are retrained with their weights saved, then
# heat-maps are drawn on held-out fold-1 patients.
RUN_GRADCAM      = True
GRADCAM_BACKBONE = "efficientnet_b2"
GRADCAM_MODS     = ["FLAIR", "T1wCE"]
DEDUP_THRESHOLD   = 0.95       # cosine similarity of FLAIR+T2 tumour crops
TIME_BUDGET_H     = 11.0       # stop taking new jobs after this (Kaggle cap is 12 h)
# 0 = images are augmented inside each GPU process. Extra loader processes are
# forked from a CUDA process with live threads, which can freeze silently on
# Python 3.13 -- the augmentation is cheap (~1 s/epoch), so they are not worth it.
LOADER_WORKERS    = 0
STALL_MINUTES     = 45         # watchdog: stop workers if no model finishes for this long
PROGRESS_MINUTES  = 5          # print a progress line this often during training

# Environment overrides (used only for local testing; leave alone on Kaggle)
ON_KAGGLE  = os.path.isdir("/kaggle")
INPUT_ROOT = os.environ.get("MERGED_INPUT_ROOT", "/kaggle/input" if ON_KAGGLE else ".")
WORK       = os.environ.get("MERGED_WORK", "/kaggle/working" if ON_KAGGLE else "./merged_work")
DEVICE     = os.environ.get("MERGED_DEVICE", "cuda")
PRETRAINED = os.environ.get("MERGED_PRETRAINED", "1") == "1"
LOADER_WORKERS = int(os.environ.get("MERGED_LOADER_WORKERS", LOADER_WORKERS))
if os.environ.get("MERGED_QUICK") == "1":
    QUICK_TEST = True

if QUICK_TEST:
    EPOCHS, MODALITIES, SEEDS = 2, ["FLAIR", "T2w"], [42]
    BACKBONES, RUN_CONTROL = ["efficientnet_b0"], False
    BASELINE_BACKBONES = ["resnet18", "simple_cnn"]
    ABLATIONS, RUN_CONTROL_B2 = ["single", "randinit"], False
    TIME_BUDGET_H = 3.0   # image preparation alone can take ~45 min on a fresh session

# Input configuration of the proposed model: the largest-tumour slice only (one
# slice, repeated to the 3 input channels). Every experiment -- proposed model,
# baselines, controls, external validation, ablation, Grad-CAM -- uses it.
# Job names carry an "SS" suffix; the proposed ensemble reuses the run-3 models
# trained with this input ("ablsingle_*" jobs).
SINGLE_SLICE = True
if SINGLE_SLICE:
    ABLATIONS = [a for a in ABLATIONS if a != "single"]
    EXTERNAL_BACKBONES = list(BACKBONES)        # external test of the full B0 + B2 ensemble
else:
    EXTERNAL_BACKBONES = [EXTERNAL_BACKBONE]

OUT_DIR    = os.path.join(WORK, "outputs_merged" + ("_quick" if QUICK_TEST else ""))
JOB_DIR    = os.path.join(OUT_DIR, "jobs")
WEIGHT_DIR = os.path.join(OUT_DIR, "cam_weights")
CACHE_ROOT = os.path.join(WORK, "cache_merged")
TMP        = os.path.join(WORK, "tmp_extract") if not ON_KAGGLE else "/kaggle/temp/brats"


def seed_all(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except Exception:
        pass


def short(backbone):
    return backbone.replace("efficientnet_", "")


def make_model(name, pretrained):
    """timm backbone with a 1-logit head, or the basic from-scratch CNN baseline."""
    import torch.nn as nn, timm
    if name == "simple_cnn":
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o),
                                 nn.ReLU(inplace=True), nn.MaxPool2d(2))
        return nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256),
                             nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, 1))
    return timm.create_model(name, pretrained=pretrained, num_classes=1, in_chans=3)


def job_name(kind, backbone, mod, seed, fold):
    if SINGLE_SLICE:
        # the proposed models already exist from run 3 under "ablsingle"
        kind = "ablsingle" if kind == "cv" else f"{kind}SS"
    return f"{kind}_{short(backbone)}_{mod}_s{seed}_f{fold}_e{EPOCHS}"


# ------------------------------- discovery -----------------------------------
_ALL_FILES = None


# RSNA DICOM series folders (~350,000 .dcm files). Only train_labels.csv is used
# from RSNA, so these folders are skipped -- walking them alone takes minutes.
SKIP_DIRS = {"FLAIR", "T1w", "T1wCE", "T2w"}


def all_input_files():
    """One walk over the inputs, reused by every search."""
    global _ALL_FILES
    if _ALL_FILES is None:
        t0 = time.time(); _ALL_FILES = []
        print("scanning inputs ...")
        for root, dirs, files in os.walk(INPUT_ROOT, followlinks=True):
            dirs[:] = [d for d in dirs if d not in SKIP_DIRS
                       and not os.path.abspath(os.path.join(root, d)).startswith(os.path.abspath(WORK))]
            _ALL_FILES += [os.path.join(root, f) for f in files]
        print(f"scanned {len(_ALL_FILES):,} input files in {time.time()-t0:.0f}s")
    return _ALL_FILES


def find_files(pattern):
    return sorted(f for f in all_input_files() if fnmatch.fnmatch(os.path.basename(f), pattern))


def list_inputs():
    print("Inputs under", INPUT_ROOT)
    for d in sorted(glob.glob(os.path.join(INPUT_ROOT, "*"))):
        print("   ", d)


def _sid(name):
    m = re.search(r"BraTS2021_(\d+)", name); return int(m.group(1)) if m else None


def _brats_key(name):
    n = name.lower()
    for k in ["_seg.nii.gz", "_flair.nii.gz", "_t1ce.nii.gz", "_t1.nii.gz", "_t2.nii.gz"]:
        if n.endswith(k): return k.strip("_").replace(".nii.gz", "")
    return None


def find_brats_source():
    tar = find_files("BraTS2021_Training_Data.tar") or find_files("*Training*Data*.tar")
    if not tar:
        tar = [t for t in find_files("*.tar") if "brats" in t.lower() or "training" in t.lower()]
    if tar:
        extra = find_files("BraTS2021_00*.tar")
        print("BraTS source: TAR ->", tar[0], f"(+{len(extra)} shards)")
        return ("tar", tar[0], extra)
    dirs = {}
    for s in find_files("BraTS2021_*_seg.nii.gz"):
        sid = _sid(os.path.basename(s))
        if sid is not None: dirs.setdefault(sid, os.path.dirname(s))
    if dirs:
        print(f"BraTS source: FOLDERS -> {len(dirs)} patients")
        return ("dirs", dirs, [])
    return (None, None, [])


# ------------------------------- image cache ---------------------------------
def robust_norm(x):
    x = x.astype(np.float32); nz = x[x > 0]
    if nz.size == 0: return np.zeros_like(x)
    lo, hi = np.percentile(nz, [1, 99]); x = np.clip(x, lo, hi)
    return (x - lo) / (hi - lo + 1e-6)


def _reorient(img, axcodes):
    from nibabel.orientations import io_orientation, axcodes2ornt, ornt_transform
    cur, tgt = io_orientation(img.affine), axcodes2ornt(axcodes)
    if np.array_equal(cur, tgt): return img
    return img.as_reoriented(ornt_transform(cur, tgt))


def build_stack(files, out_dir, name, reorient_to=None, use_crop=True, slice_mode="tumor"):
    """Tumour-focused 2.5D stack per modality -- the paper's _build_one.
    files: {"seg","flair","t1","t1ce","t2"} -> path.
    reorient_to=None keeps BraTS volumes byte-identical to the paper; extra
    cohorts are reoriented to the BraTS axis codes and their slice gap is
    rescaled to the same physical distance. Returns tumour voxel count or None.
    Ablation variants: use_crop=False keeps the whole slice; slice_mode="center"
    takes the geometric centre slice instead of the largest-tumour slice."""
    import nibabel as nib, cv2
    need = ["seg"] + [SUFFIX[m] for m in MODALITIES]
    if any(k not in files for k in need): return None
    try:
        segi = nib.load(files["seg"])
        if reorient_to is not None: segi = _reorient(segi, reorient_to)
        tumor = np.asarray(segi.get_fdata()) > 0
        if tumor.ndim != 3 or tumor.sum() == 0: return None
        gap = SLICE_GAP
        if reorient_to is not None:
            gap = max(1, int(round(SLICE_GAP * 1.0 / float(segi.header.get_zooms()[2]))))
        z_tumor = int(np.argmax(tumor.sum(axis=(0, 1))))
        ys, xs = np.where(tumor.any(axis=2))
        y0, y1 = max(0, ys.min() - CROP_MARGIN), ys.max() + CROP_MARGIN
        x0, x1 = max(0, xs.min() - CROP_MARGIN), xs.max() + CROP_MARGIN
        off = [(i - N_SLICES // 2) * gap for i in range(N_SLICES)]
        stacks = {}
        for mod in MODALITIES:
            img = nib.load(files[SUFFIX[mod]])
            if reorient_to is not None: img = _reorient(img, reorient_to)
            vol = np.asarray(img.get_fdata())
            if vol.ndim == 4: vol = vol[..., 0]
            if vol.shape != tumor.shape: return None
            d = vol.shape[2]; chans = []
            base_z = d // 2 if slice_mode == "center" else z_tumor
            for o in off:
                zi = min(max(0, base_z + o), d - 1)
                sl = vol[y0:y1, x0:x1, zi] if use_crop else vol[:, :, zi]
                chans.append(cv2.resize(robust_norm(sl), (IMG_SIZE, IMG_SIZE)))
            stacks[mod] = (np.stack(chans, -1) * 255).astype(np.uint8)
        for mod, arr in stacks.items():
            np.save(os.path.join(out_dir, f"{mod}_{name}.npy"), arr)
        return int(tumor.sum())
    except Exception as e:
        print(f"   skip {name}: {type(e).__name__}: {e}")
        return None


def brats_axcodes(source):
    """Axis codes of the BraTS volumes; extra cohorts are aligned to these."""
    import nibabel as nib
    kind, data, _ = source
    try:
        if kind == "dirs":
            folder = next(iter(data.values()))
            seg = glob.glob(os.path.join(folder, "*_seg.nii.gz"))[0]
            return tuple(nib.aff2axcodes(nib.load(seg).affine))
        if kind == "tar":
            os.makedirs(TMP, exist_ok=True)
            with tarfile.open(data) as tf:
                for m in tf:
                    if m.isfile() and m.name.lower().endswith("_seg.nii.gz"):
                        dst = os.path.join(TMP, "axref_seg.nii.gz")
                        with tf.extractfile(m) as src, open(dst, "wb") as out:
                            shutil.copyfileobj(src, out)
                        codes = tuple(nib.aff2axcodes(nib.load(dst).affine))
                        os.remove(dst); return codes
    except Exception as e:
        print("   could not read BraTS orientation:", e)
    print("   WARNING: assuming BraTS orientation ('L','P','S')")
    return ("L", "P", "S")


def reuse_cache_from_inputs(dst, names):
    """Copy a cache folder from a previous run's output (attached as input)."""
    if os.path.isfile(os.path.join(dst, "done.flag")): return
    for flag in find_files("done.flag"):
        if os.path.basename(os.path.dirname(flag)) in names:
            print(f"reusing cache from input: {os.path.dirname(flag)}")
            shutil.copytree(os.path.dirname(flag), dst, dirs_exist_ok=True); return


def build_brats_cache(needed, source, variant="main"):
    out = os.path.join(CACHE_ROOT, "brats" if variant == "main" else f"brats_{variant}")
    if variant == "main":
        reuse_cache_from_inputs(out, {"brats", "imgcache", "imgcache_brats"})
    opts = CACHE_VARIANTS.get(variant, {})
    os.makedirs(out, exist_ok=True)
    flag = os.path.join(out, "done.flag")
    if os.path.isfile(flag):
        cov = set(int(x) for x in open(flag).read().split())
        print(f"reusing BraTS {variant} cache ({len(cov)} patients)"); return out, cov
    t0 = time.time(); covered = set(); kind, data, extra = source
    print(f"preparing BraTS images ({variant}) for {len(needed)} patients (progress every 50) ...")

    def add(sid):
        covered.add(sid)
        if len(covered) % 50 == 0:
            print(f"   BraTS: {len(covered)}/{len(needed)} patients ({(time.time()-t0)/60:.1f} min)")

    if kind == "tar":
        os.makedirs(TMP, exist_ok=True)
        for path in [data, *extra]:
            cur_id, cur = None, {}
            with tarfile.open(path) as tf:
                for m in tf:
                    if not m.isfile(): continue
                    sid = _sid(m.name)
                    if sid not in needed: continue
                    k = _brats_key(m.name)
                    if k is None: continue
                    if sid != cur_id and cur_id is not None:
                        if build_stack(cur, out, cur_id, **opts) is not None: add(cur_id)
                        for f in cur.values():
                            try: os.remove(f)
                            except OSError: pass
                        cur = {}
                    cur_id = sid
                    dst = os.path.join(TMP, os.path.basename(m.name))
                    with tf.extractfile(m) as src, open(dst, "wb") as fo:
                        shutil.copyfileobj(src, fo)
                    cur[k] = dst
                if cur_id is not None and build_stack(cur, out, cur_id, **opts) is not None:
                    add(cur_id)
    elif kind == "dirs":
        for sid, folder in data.items():
            if sid not in needed: continue
            files = {}
            for f in glob.glob(os.path.join(folder, "*.nii.gz")):
                k = _brats_key(f)
                if k: files[k] = f
            if build_stack(files, out, sid, **opts) is not None: add(sid)
    if not covered:
        sys.exit("No BraTS images could be cached -- check the BraTS 2021 input.")
    open(flag, "w").write(" ".join(map(str, sorted(covered))))
    print(f"built BraTS cache: {len(covered)}/{len(needed)} patients in {time.time()-t0:.0f}s")
    return out, covered


# ---- extra cohorts -------------------------------------------------------------
def parse_mgmt(v):
    s = str(v).strip().lower()
    if s in ("", "nan", "none"): return None
    if s.startswith("unmethyl") or s in ("negative", "neg", "0", "0.0", "no"): return 0
    if s.startswith("methyl") or s in ("positive", "pos", "1", "1.0", "yes"): return 1
    return None                   # indeterminate / not available


def pick_col(df, *needles):
    for c in df.columns:
        if all(n in c.lower() for n in needles): return c
    return None


def _digits(v):
    m = re.search(r"(\d+)\D*$", str(v)); return int(m.group(1)) if m else None


def site_key(site, sid):
    return _digits(sid) if site == "UCSF" else str(sid).strip().upper()


def discover_site_files(site):
    """{subject_id: {seg, flair, t1, t1ce, t2}} for one extra cohort."""
    subs = {}
    if site == "UPENN":
        rx = re.compile(r"^(UPENN-GBM-\d+_\d+)_(.+)\.nii(?:\.gz)?$", re.I)
        for f in all_input_files():
            m = rx.match(os.path.basename(f))
            if not m: continue
            sid, tag = m.group(1).upper(), m.group(2).lower()
            d = subs.setdefault(sid, {})
            if "seg" in tag:
                # prefer the manual segmentation over the automated one
                if "automated" not in tag or "seg" not in d: d["seg"] = f
            else:
                key = {"flair": "flair", "t1": "t1", "t1gd": "t1ce", "t1ce": "t1ce", "t2": "t2"}.get(tag)
                if key: d[key] = f
    elif site == "UCSF":
        rx = re.compile(r"^(UCSF-PDGM-\d+)_(.+)\.nii(?:\.gz)?$", re.I)
        for f in all_input_files():
            m = rx.match(os.path.basename(f))
            if not m: continue
            sid, tag = m.group(1).upper(), m.group(2).lower()
            d = subs.setdefault(sid, {})
            if tag == "tumor_segmentation":
                d["seg"] = f; continue
            bias = tag.endswith("_bias")
            key = {"flair": "flair", "t1": "t1", "t1c": "t1ce", "t2": "t2"}.get(tag.replace("_bias", ""))
            if key and (not bias or key not in d):   # prefer the non-bias-corrected file
                d[key] = f
    return subs


def load_site_labels(site):
    """DataFrame[key, subject_id, MGMT_value, brats21_id]."""
    rows = []
    if site == "UPENN":
        lab = find_files("upenn_mgmt_labels.csv")
        if lab:
            d = pd.read_csv(lab[0])
            for _, r in d.iterrows():
                y = parse_mgmt(r["MGMT_value"])
                if y is not None: rows.append((site_key(site, r["subject_id"]), r["subject_id"], y, None))
        else:
            clin = find_files("UPENN-GBM*clinical*.csv")
            if clin:
                d = pd.read_csv(clin[0]); idc = pick_col(d, "id"); mc = pick_col(d, "mgmt")
                for _, r in d.iterrows():
                    y = parse_mgmt(r[mc])
                    if y is not None: rows.append((site_key(site, r[idc]), r[idc], y, None))
    elif site == "UCSF":
        meta = find_files("UCSF-PDGM*metadata*.csv") or find_files("*PDGM*meta*.csv")
        if meta:
            d = pd.read_csv(meta[0])
            idc = pick_col(d, "id") if "ID" not in d.columns else "ID"
            mc = pick_col(d, "mgmt", "status") or pick_col(d, "mgmt")
            ic = pick_col(d, "mgmt", "index")
            gc = pick_col(d, "grade")
            bc = pick_col(d, "brats", "id") or pick_col(d, "brats")
            for _, r in d.iterrows():
                y = parse_mgmt(r[mc])
                if y is None: continue
                if UCSF_MGMT_INDEX_CUTOFF is not None and ic is not None:
                    v = pd.to_numeric(r[ic], errors="coerce")
                    if pd.notna(v): y = int(v >= UCSF_MGMT_INDEX_CUTOFF)
                if EXTRA_GRADE4_ONLY and gc is not None and _digits(r[gc]) not in (4, None):
                    continue
                b = _digits(r[bc]) if bc is not None and pd.notna(r[bc]) else None
                rows.append((site_key(site, r[idc]), r[idc], y, b))
    return pd.DataFrame(rows, columns=["key", "subject_id", "MGMT_value", "brats21_id"])


def build_site_cache(site, axcodes):
    out = os.path.join(CACHE_ROOT, site.lower())
    reuse_cache_from_inputs(out, {site.lower()})
    meta_p = os.path.join(out, "meta.csv")
    if os.path.isfile(os.path.join(out, "done.flag")) and os.path.isfile(meta_p):
        meta = pd.read_csv(meta_p)
        print(f"reusing {site} cache ({len(meta)} patients)"); return out, meta
    labels = load_site_labels(site)
    if labels.empty:
        print(f"{site}: no MGMT labels found -> site skipped"); return None, None
    files = discover_site_files(site)
    fkey = {site_key(site, s): (s, f) for s, f in files.items()}
    jobs, rows = [], []
    for i, r in enumerate(labels.drop_duplicates("key").itertuples()):
        if r.key in fkey:
            sid, f = fkey[r.key]
            jobs.append((f, out, str(i), axcodes))
            rows.append({"idx": i, "subject_id": sid, "MGMT_value": int(r.MGMT_value),
                         "brats21_id": r.brats21_id})
    print(f"{site}: {len(labels)} labelled, {len(files)} on disk, {len(jobs)} matched")
    if not jobs:
        print(f"{site}: no labelled subject has images -> site skipped"); return None, None
    os.makedirs(out, exist_ok=True)
    # plain loop (a forked process pool can freeze once OpenCV threads exist)
    t0 = time.time(); res = {}
    for k, (f, o, name, ax) in enumerate(jobs, 1):
        res[name] = build_stack(f, o, name, reorient_to=ax)
        if k % 50 == 0 or k == len(jobs):
            print(f"   {site}: {k}/{len(jobs)} patients prepared ({(time.time()-t0)/60:.1f} min)")
    meta = pd.DataFrame([dict(r, tumor_vox=res[str(r["idx"])]) for r in rows
                         if res.get(str(r["idx"])) is not None])
    if meta.empty:
        print(f"{site}: no subject could be cached -> site skipped"); return None, None
    meta["site"] = site
    meta.to_csv(meta_p, index=False)
    open(os.path.join(out, "done.flag"), "w").write(str(len(meta)))
    print(f"{site}: cached {len(meta)} patients (pos rate {meta.MGMT_value.mean():.3f}) "
          f"in {time.time()-t0:.0f}s")
    return out, meta


def build_site_variant_cache(site, axcodes, variant, main_meta):
    """Ablation copy of an extra cohort's cache (whole slice / centre slice) for
    exactly the patients of the main cache, with the same patient numbering."""
    out = os.path.join(CACHE_ROOT, f"{site.lower()}_{variant}")
    flag = os.path.join(out, "done.flag")
    if os.path.isfile(flag):
        print(f"reusing {site} {variant} cache"); return out
    labels, files = load_site_labels(site), discover_site_files(site)
    fkey = {site_key(site, s): f for s, f in files.items()}
    want = set(int(i) for i in main_meta.idx)
    os.makedirs(out, exist_ok=True); t0 = time.time(); k = 0
    for i, r in enumerate(labels.drop_duplicates("key").itertuples()):
        if i not in want or r.key not in fkey: continue
        build_stack(fkey[r.key], out, str(i), reorient_to=axcodes, **CACHE_VARIANTS[variant]); k += 1
        if k % 50 == 0 or k == len(want):
            print(f"   {site} ({variant}): {k}/{len(want)} patients prepared ({(time.time()-t0)/60:.1f} min)")
    open(flag, "w").write(str(k))
    return out


def cache_complete(cache_dir, names):
    """True if every patient has all modality files in this cache."""
    return all(os.path.isfile(os.path.join(cache_dir, f"{m}_{n}.npy")) for n in names for m in MODALITIES)


# ---- duplicate audit ---------------------------------------------------------
def dedup_audit(site, meta, site_cache, brats_cache, eval_ids, eval_labels):
    import cv2
    def sig(cache, name):
        parts = []
        for mod in ("FLAIR", "T2w"):
            p = os.path.join(cache, f"{mod}_{name}.npy")
            if not os.path.isfile(p): continue
            a = cv2.resize(np.load(p).astype(np.float32).mean(-1), (32, 32))
            parts.append((a - a.mean()) / (a.std() + 1e-6))
        return np.concatenate([x.ravel() for x in parts]) if parts else None
    bsig = {i: sig(brats_cache, i) for i in eval_ids}
    bids = [i for i, v in bsig.items() if v is not None]
    B = np.stack([bsig[i] for i in bids]); B /= np.linalg.norm(B, axis=1, keepdims=True)
    eval_set = set(int(i) for i in eval_ids)
    rows = []
    for r in meta.itertuples():
        s = sig(site_cache, r.idx)
        sims = B @ (s / np.linalg.norm(s)) if s is not None else np.zeros(len(bids))
        j = int(np.argmax(sims))
        exact = pd.notna(r.brats21_id) and int(r.brats21_id) in eval_set
        matched = str(int(r.brats21_id)) if exact else bids[j]
        rows.append({"site": site, "idx": r.idx, "subject_id": r.subject_id,
                     "nearest_brats": bids[j], "max_cosine": float(sims[j]),
                     "brats21_id_match": bool(exact),
                     "image_match": bool(sims[j] > DEDUP_THRESHOLD),
                     "excluded": bool(exact or sims[j] > DEDUP_THRESHOLD),
                     "site_label": int(r.MGMT_value), "rsna_label": eval_labels.get(matched)})
    rep = pd.DataFrame(rows)
    print(f"{site} dedup: {int(rep.brats21_id_match.sum())} exact BraTS21-ID matches, "
          f"{int((rep.max_cosine > DEDUP_THRESHOLD).sum())} image matches > {DEDUP_THRESHOLD} "
          f"-> {int(rep.excluded.sum())} excluded | similarity quantiles 50/90/99% = "
          f"{np.percentile(rep.max_cosine, [50, 90, 99]).round(3).tolist()}")
    ex = rep[rep.brats21_id_match]
    if len(ex):
        print(f"   image check found {int(ex.image_match.sum())}/{len(ex)} of the known (ID) duplicates")
    dup = rep[rep.excluded]
    if len(dup):
        print(f"   label agreement with RSNA on duplicates: "
              f"{(dup.site_label == dup.rsna_label).mean():.3f} (n={len(dup)})")
    return rep


# ------------------------------- GPU worker ----------------------------------
class TrainDS:
    """Training dataset (module level so DataLoader worker processes can load it
    under any multiprocessing start method -- Python 3.13+ on Kaggle)."""
    def __init__(self, arrs, labels, tf): self.arrs, self.labels, self.tf = arrs, labels, tf
    def __len__(self): return len(self.arrs)
    def __getitem__(self, i):
        import torch
        return self.tf(image=self.arrs[i])["image"], torch.tensor(self.labels[i], dtype=torch.float32)


def worker(rank, n_workers, jobs, cfg):
    """Runs in its own process on one GPU (CUDA_VISIBLE_DEVICES = that GPU).
    Both workers walk the same job list; a lock file decides who trains what."""
    import torch, torch.nn as nn, timm, cv2
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    from torch.utils.data import Dataset, DataLoader
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedKFold

    cv2.setNumThreads(1)
    torch.set_num_threads(max(1, (os.cpu_count() or 2) // n_workers - LOADER_WORKERS))
    if DEVICE == "cuda":
        torch.cuda.set_device(0); dev = torch.device("cuda", 0)   # only this GPU is visible
        tag = f"[GPU{rank} {torch.cuda.get_device_name(0)}]"
    else:
        dev = torch.device(DEVICE); tag = f"[worker{rank} {DEVICE}]"
    use_amp = dev.type == "cuda"
    print(f"{tag} started", flush=True)

    tr_tf = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(0.10, 0.10, 20, border_mode=cv2.BORDER_CONSTANT, p=0.5),
        A.RandomBrightnessContrast(0.1, 0.1, p=0.3),
        A.Normalize((0.5,) * 3, (0.5,) * 3), ToTensorV2()])

    store, store_variant = {}, [None]
    def load(site, name, mod, variant="main"):
        # variant: "main" | ablation name. "nocrop"/"center" read their own cache;
        # single-slice input keeps only the middle slice (repeated to 3 channels).
        if variant != store_variant[0]:
            store.clear(); store_variant[0] = variant      # bound memory per worker
        cache = variant if variant in cfg["caches"] else "main"
        k = (site, name, mod)
        if k not in store:
            a = np.load(os.path.join(cfg["caches"][cache][site], f"{mod}_{name}.npy"))
            if SINGLE_SLICE or variant == "single":
                a = np.ascontiguousarray(np.repeat(a[:, :, 1:2], 3, axis=2))
            store[k] = a
        return store[k]

    def to_tensor(arrs):
        x = torch.from_numpy(np.stack(arrs)).permute(0, 3, 1, 2).float()
        return (x / 255.0 - 0.5) / 0.5

    @torch.no_grad()
    def predict(model, X, tta):
        model.eval(); out = []
        for i in range(0, len(X), 64):
            xb = X[i:i + 64].to(dev)
            with torch.autocast(dev.type, dtype=torch.float16, enabled=use_amp):
                p = torch.sigmoid(model(xb).squeeze(1).float())
                if tta:
                    p = (p + torch.sigmoid(model(torch.flip(xb, [3])).squeeze(1).float())) / 2
            out.append(torch.nan_to_num(p, nan=0.5).cpu())
        return torch.cat(out).numpy()

    def safe_auc(y, p):
        ok = np.isfinite(p)
        if ok.sum() < 2 or len(np.unique(y[ok])) < 2: return 0.5
        return roc_auc_score(y[ok], p[ok])

    b_ids = np.array(cfg["brats_ids"]); b_y = np.array(cfg["brats_y"])
    splits = {s: list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=s).split(b_ids, b_y))
              for s in cfg["seeds"]}

    claim_dir = os.path.join(JOB_DIR, "claims"); total = len(jobs)
    for job in jobs:
        if os.path.isfile(os.path.join(JOB_DIR, job["name"] + ".npz")): continue
        try:
            os.close(os.open(os.path.join(claim_dir, job["name"]), os.O_CREAT | os.O_EXCL | os.O_WRONLY))
        except FileExistsError:
            continue                      # the other GPU took this job
        if time.time() > cfg["deadline"]:
            print(f"{tag} time budget reached -> skipping {job['name']}", flush=True); continue
        t0 = time.time()
        try:
            mod, seed, fold = job["modality"], job["seed"], job["fold"]
            tr, va = splits[seed][fold]
            tr_keys = [("BRATS", str(i)) for i in b_ids[tr]]; tr_y = list(b_y[tr])
            for site in job["train_sites"]:
                tr_keys += [(site, str(i)) for i in cfg["sites"][site]["ids"]]
                tr_y += list(cfg["sites"][site]["y"])
            variant = job.get("variant", "main")
            tr_arrs = [load(s, n, mod, variant) for s, n in tr_keys]
            Xva = to_tensor([load("BRATS", str(i), mod, variant) for i in b_ids[va]]); yva = b_y[va]
            Xte = None
            if job.get("test_site"):
                ts = cfg["sites"][job["test_site"]]
                Xte = to_tensor([load(job["test_site"], str(i), mod, variant) for i in ts["ids"]])

            print(f"{tag} start {job['name']} ({len(tr_arrs)} training images)", flush=True)
            seed_all(seed + fold)
            # num_workers=0: no loader processes (see LOADER_WORKERS); a timeout
            # turns any loader stall into an error instead of a silent hang
            dl = DataLoader(TrainDS(tr_arrs, np.array(tr_y, np.float32), tr_tf), batch_size=BATCH,
                            shuffle=True, drop_last=True, num_workers=LOADER_WORKERS,
                            persistent_workers=LOADER_WORKERS > 0,
                            timeout=600 if LOADER_WORKERS > 0 else 0)
            model = make_model(job["backbone"], PRETRAINED and variant != "randinit").to(dev)
            optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=job["epochs"])
            try:
                scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
            except (AttributeError, TypeError):
                scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
            crit = nn.BCEWithLogitsLoss()
            best, best_ep, best_state = -1.0, -1, None
            for ep in range(job["epochs"]):
                model.train()
                for xb, yb in dl:
                    xb, yb = xb.to(dev, non_blocking=True), yb.to(dev, non_blocking=True)
                    with torch.autocast(dev.type, dtype=torch.float16, enabled=use_amp):
                        loss = crit(model(xb).squeeze(1), yb)
                    if not torch.isfinite(loss):
                        optim.zero_grad(set_to_none=True); continue
                    optim.zero_grad(set_to_none=True); scaler.scale(loss).backward()
                    scaler.unscale_(optim); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optim); scaler.update()
                # paper protocol: epoch chosen on the held-out BraTS fold (no TTA)
                va_auc = safe_auc(yva, predict(model, Xva, tta=False)); sched.step()
                if va_auc > best:
                    best, best_ep = va_auc, ep
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            res = {"va_idx": va, "last_va": predict(model, Xva, tta=True)}
            if Xte is not None: res["last_te"] = predict(model, Xte, tta=True)
            if best_state is not None: model.load_state_dict(best_state)
            res["best_va"] = predict(model, Xva, tta=True)
            if Xte is not None: res["best_te"] = predict(model, Xte, tta=True)
            res["best_epoch"] = best_ep; res["best_sel_auc"] = best
            if job.get("save_weights") and best_state is not None:     # for Grad-CAM
                os.makedirs(WEIGHT_DIR, exist_ok=True)
                torch.save(best_state, os.path.join(WEIGHT_DIR, job["name"] + ".pt"))
            tmp = os.path.join(JOB_DIR, job["name"] + ".tmp.npz")
            np.savez(tmp, **res); os.replace(tmp, os.path.join(JOB_DIR, job["name"] + ".npz"))
            del model, optim, dl
            if DEVICE == "cuda": torch.cuda.empty_cache()
            done = len(glob.glob(os.path.join(JOB_DIR, "*.npz"))) - cfg["done_before"]
            print(f"{tag} {done:3d}/{total}  {job['name']:<38s} sel-AUC {best:.3f} @ep{best_ep+1:<2d} "
                  f"({(time.time()-t0)/60:.1f} min)", flush=True)
        except Exception:
            print(f"{tag} FAILED {job['name']}\n{traceback.format_exc()}", flush=True)
    print(f"{tag} finished", flush=True)


def _script_path():
    """Path of this script, so each GPU can run it as its own process. When the
    code was pasted into a notebook cell, the cell source is written to a file."""
    p = globals().get("__file__")
    if p and os.path.isfile(p) and p.endswith(".py"):
        return os.path.abspath(p)
    import linecache
    src = "".join(linecache.getlines(worker.__code__.co_filename))
    # a cell that also holds !shell lines cannot run as a plain script
    if "def worker(" not in src or re.search(r"^\s*get_ipython\(\)", src, re.M):
        sys.exit("Could not find the script file. Run it as:\n"
                 "  !python /kaggle/working/kaggle_merged_training.py")
    p = os.path.join(WORK, "_merged_runtime.py")
    with open(p, "w") as fh: fh.write(src)
    return p


def _json_default(o):
    return o.item() if hasattr(o, "item") else str(o)


def run_jobs(jobs, cfg):
    import subprocess, threading
    os.makedirs(JOB_DIR, exist_ok=True)
    pending = [j for j in jobs if not os.path.isfile(os.path.join(JOB_DIR, j["name"] + ".npz"))]
    print(f"\n{len(jobs)} models in total, {len(jobs)-len(pending)} already done, "
          f"{len(pending)} to train")
    if not pending: return
    if DEVICE == "cuda":
        import torch; n_workers = torch.cuda.device_count()
    else:
        n_workers = 2
    print(f"training on {n_workers} device(s) in parallel\n", flush=True)
    claim_dir = os.path.join(JOB_DIR, "claims")
    shutil.rmtree(claim_dir, ignore_errors=True); os.makedirs(claim_dir)
    cfg = dict(cfg, done_before=len(glob.glob(os.path.join(JOB_DIR, "*.npz"))))
    spec = os.path.join(JOB_DIR, "worker_spec.json")
    with open(spec, "w") as fh:
        json.dump({"jobs": pending, "cfg": cfg, "n_workers": n_workers}, fh, default=_json_default)
    script = _script_path()
    visible = os.environ.get("CUDA_VISIBLE_DEVICES")
    gpu_ids = visible.split(",") if visible else [str(i) for i in range(n_workers)]
    procs, relays = [], []
    for r in range(n_workers):
        # weights were downloaded by main(); workers read the local copy (no network calls)
        env = dict(os.environ, PYTHONUNBUFFERED="1", HF_HUB_OFFLINE="1")
        if DEVICE == "cuda": env["CUDA_VISIBLE_DEVICES"] = gpu_ids[r]
        p = subprocess.Popen([sys.executable, "-u", script, "--worker", str(r), spec], env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        t = threading.Thread(target=lambda p=p: [print(l, end="", flush=True) for l in p.stdout])
        t.start(); procs.append(p); relays.append(t)

    # progress line every PROGRESS_MINUTES; stop the workers if nothing finishes
    # for STALL_MINUTES, so a frozen run still writes its results and ends
    t0 = time.time(); done_before = cfg["done_before"]
    finished = lambda: len(glob.glob(os.path.join(JOB_DIR, "*.npz"))) - done_before
    last_n, last_change, last_print = 0, time.time(), time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(15)
        n = finished()
        if n != last_n: last_n, last_change = n, time.time()
        if time.time() - last_print >= PROGRESS_MINUTES * 60:
            last_print = time.time(); el = (time.time() - t0) / 3600
            eta = f", about {el / n * (len(pending) - n):.1f} h left" if n else ""
            running = [os.path.basename(c) for c in glob.glob(os.path.join(claim_dir, "*"))
                       if not os.path.isfile(os.path.join(JOB_DIR, os.path.basename(c) + ".npz"))]
            print(f"[progress] {n}/{len(pending)} models finished | {el:.2f} h elapsed{eta} "
                  f"| training now: {', '.join(running) or '-'}", flush=True)
        if time.time() - last_change > STALL_MINUTES * 60:
            print(f"\nSTALL: no model finished in {STALL_MINUTES} min -- stopping the GPU workers. "
                  f"Finished models are saved; send this log to Claude.", flush=True)
            for p in procs:
                if p.poll() is None: p.kill()
            break
    for p in procs: p.wait()
    for t in relays: t.join(timeout=30)
    for r, p in enumerate(procs):
        if p.returncode != 0:
            print(f"WARNING: GPU worker {r} exited with code {p.returncode} (see its messages above)")


def _worker_cli(rank, spec_path):
    with open(spec_path) as fh: spec = json.load(fh)
    worker(rank, spec["n_workers"], spec["jobs"], spec["cfg"])


def restore_jobs_from_inputs():
    """Resume: copy finished job files from a previous run attached as input."""
    os.makedirs(JOB_DIR, exist_ok=True); n = 0
    for f in all_input_files():
        parent = os.path.dirname(f)
        if f.endswith(".npz") and os.path.basename(parent) == "jobs" \
                and os.path.basename(os.path.dirname(parent)) == os.path.basename(OUT_DIR):
            dst = os.path.join(JOB_DIR, os.path.basename(f))
            if not os.path.isfile(dst): shutil.copy(f, dst); n += 1
    if n: print(f"resumed {n} finished models from inputs")


# ------------------------------- statistics ----------------------------------
def rankise(p):
    from scipy.stats import rankdata
    return rankdata(p) / len(p)


def auc_ci(y, p, B=2000, seed=42):
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(seed); n = len(y); b = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        if len(np.unique(y[i])) > 1: b.append(roc_auc_score(y[i], p[i]))
    return roc_auc_score(y, p), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


def delong(y, p1, p2):
    """DeLong test for two correlated AUCs on the same patients. Returns (z, p)."""
    from scipy.stats import norm
    y = np.asarray(y).astype(int); pos, neg = y == 1, y == 0
    def comps(p):
        d = p[pos][:, None] - p[neg][None, :]
        psi = (d > 0) + 0.5 * (d == 0)
        return psi.mean(1), psi.mean(0), psi.mean()
    a10, a01, a = comps(np.asarray(p1)); b10, b01, b = comps(np.asarray(p2))
    S = np.cov(np.vstack([a10, b10])) / pos.sum() + np.cov(np.vstack([a01, b01])) / neg.sum()
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    if var <= 0: return 0.0, 1.0
    z = (a - b) / np.sqrt(var); return float(z), float(2 * norm.sf(abs(z)))


def mcc_youden(y, p):
    from sklearn.metrics import roc_curve, matthews_corrcoef
    fpr, tpr, thr = roc_curve(y, p); t = thr[np.argmax(tpr - fpr)]
    return matthews_corrcoef(y, (p >= t).astype(int))


def skill_weighted(per_mod, y):
    from sklearn.metrics import roc_auc_score
    w = np.array([max(roc_auc_score(y, per_mod[m]) - 0.5, 1e-3) for m in MODALITIES]); w /= w.sum()
    return sum(w[i] * rankise(per_mod[m]) for i, m in enumerate(MODALITIES))


def assemble_cv(kind, backbone, n, which):
    """Seed-averaged OOF per modality for a cv/control group. None if incomplete."""
    per_mod = {}
    for mod in MODALITIES:
        acc = np.zeros(n)
        for seed in SEEDS:
            oof = np.full(n, np.nan)
            for fold in range(N_FOLDS):
                f = os.path.join(JOB_DIR, job_name(kind, backbone, mod, seed, fold) + ".npz")
                if not os.path.isfile(f): return None
                d = np.load(f); oof[d["va_idx"]] = d[which]
            acc += oof / len(SEEDS)
        per_mod[mod] = acc
    return per_mod


def assemble_ext(site, backbone, n_test, which):
    per_mod = {}
    for mod in MODALITIES:
        preds = []
        for seed in SEEDS:
            for fold in range(N_FOLDS):
                f = os.path.join(JOB_DIR, job_name(f"ext{site}", backbone, mod, seed, fold) + ".npz")
                if not os.path.isfile(f): return None
                preds.append(np.load(f)[which])
        per_mod[mod] = np.mean(preds, axis=0)
    return per_mod


def load_prior(df):
    """Earlier OOF predictions attached as inputs, aligned to the cohort.
    These were all trained with a different input configuration, so they are
    not reported when SINGLE_SLICE is on."""
    found = {}
    if SINGLE_SLICE:
        return found
    names = {"oof_predictions.csv": "Paper headline B0 (BraTS only)",
             "oof_improve_b1_3sl.csv": "Prior B1 (BraTS only)",
             "oof_improve_b2_3sl.csv": "Prior B2 (BraTS only)",
             "oof_improve_b0_5sl.csv": "Prior B0 5-slice (BraTS only)",
             "oof_ablation_nocrop.csv": "Prior B0 no-crop (BraTS only)"}
    for fn, label in names.items():
        hits = [h for h in find_files(fn) if "outputs_merged" not in h]
        if not hits: continue
        d = pd.read_csv(hits[0]); m = dict(zip(d.BraTS21ID.astype(int), d.pred.astype(float)))
        p = np.array([m.get(int(i), np.nan) for i in df.BraTS21ID], float)
        if np.isnan(p).any():
            print(f"   prior {fn}: {int(np.isnan(p).sum())} patients missing -> not used"); continue
        found[label] = p
    return found


# ------------------------------- reporting ------------------------------------
def report(df, y, fold42, groups, prior, external, cohort):
    from sklearn.metrics import roc_auc_score
    ref_name = "Paper headline B0 (BraTS only)"
    ref = prior.get(ref_name)
    ctrl = groups.get("Control B0 (BraTS only, this code)", {}).get("best")
    merged = [n for n in groups if n.startswith("Merged")]
    # baselines are compared with the primary ensemble AND with the best single
    # merged model (single-vs-single, so the ensemble is not the only reference)
    primary = (np.mean([rankise(groups[n]["best"]) for n in merged], 0) if len(merged) >= 2
               else (groups[merged[0]]["best"] if merged else None))
    single = max(merged, key=lambda n: roc_auc_score(y, groups[n]["best"])) if merged else None

    rows, oof_out = [], {}
    def add(name, p, strict=None):
        a, lo, hi = auc_ci(y, p)
        row = {"Model": name, "AUC (95% CI)": f"{a:.3f} ({lo:.3f}-{hi:.3f})",
               "AUC strict": f"{roc_auc_score(y, strict):.3f}" if strict is not None else "-",
               "MCC": round(mcc_youden(y, p), 3)}
        if ref is not None and name != ref_name:
            z, pv = delong(y, p, ref)
            row["dAUC vs paper"] = f"{a - roc_auc_score(y, ref):+.3f}"
            row["DeLong p vs paper"] = f"{pv:.4f}"
        if ctrl is not None and "Merged" in name:
            z, pv = delong(y, p, ctrl)
            row["dAUC vs control"] = f"{a - roc_auc_score(y, ctrl):+.3f}"
            row["DeLong p vs control"] = f"{pv:.4f}"
        if name.startswith("Baseline") and primary is not None:
            z, pv = delong(y, primary, p)
            row["primary - baseline"] = f"{roc_auc_score(y, primary) - a:+.3f}"
            row["DeLong p (primary vs baseline)"] = f"{pv:.4f}"
            sp = groups[single]["best"]; z, pv = delong(y, sp, p)
            row[f"{single.split()[1]} - baseline"] = f"{roc_auc_score(y, sp) - a:+.3f}"
            row[f"DeLong p ({single.split()[1]} vs baseline)"] = f"{pv:.4f}"
        for f in range(N_FOLDS):
            m = fold42 == f; row[f"Fold {f+1}"] = round(roc_auc_score(y[m], p[m]), 3)
        rows.append(row); oof_out[name] = p

    if ref is not None: add(ref_name, ref)
    for name, g in groups.items(): add(name, g["best"], g["last"])
    merged = [n for n in groups if n.startswith("Merged")]
    if len(merged) >= 2:
        add("Merged ENS (" + " + ".join(n.split()[1] for n in merged) + ")  <- primary",
            np.mean([rankise(groups[n]["best"]) for n in merged], 0),
            np.mean([rankise(groups[n]["last"]) for n in merged], 0))
    if merged and prior:
        comps = [groups[n]["best"] for n in merged] + list(prior.values())
        add(f"Merged + prior EfficientNet runs ({len(comps)} models)",
            np.mean([rankise(p) for p in comps], 0))
    for label, p in prior.items():
        if label != ref_name: add(label, p)

    tbl = pd.DataFrame(rows).fillna("-")
    tbl.to_csv(os.path.join(OUT_DIR, "results_table.csv"), index=False)
    for name, p in oof_out.items():
        slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")[:60]
        pd.DataFrame({"BraTS21ID": df.BraTS21ID.values, "MGMT_value": y, "pred": p,
                      "fold": fold42}).to_csv(os.path.join(OUT_DIR, f"oof_merged_{slug}.csv"), index=False)

    print("\n" + "=" * 110)
    print(f"RESULTS on the same {len(y)} RSNA-MICCAI + BraTS 2021 patients as the paper")
    print("=" * 110)
    print(tbl.to_string(index=False))
    print("-" * 110)
    print("AUC = paper protocol (epoch picked on the held-out fold, as in the paper).")
    print("AUC strict = final epoch, nothing tuned on the held-out fold (no optimism).")

    ext_tbl = None
    if external:
        ext_rows = []
        for site, d in external.items():
            a, lo, hi = auc_ci(d["y"], d["best"])
            ext_rows.append({"Held-out site": site, "n": len(d["y"]),
                             "pos rate": round(float(np.mean(d["y"])), 3),
                             "AUC (95% CI)": f"{a:.3f} ({lo:.3f}-{hi:.3f})",
                             "AUC strict": round(roc_auc_score(d["y"], d["last"]), 3),
                             **{m: round(roc_auc_score(d["y"], d["per_mod"][m]), 3) for m in MODALITIES}})
            # sensitivity check: leave out kept patients that look similar (0.90-0.95)
            # to a BraTS patient, in case any of them is an undetected duplicate
            keep = ~np.isin(np.array(d["subject_id"]), list(d["borderline"]))
            if keep.sum() < len(keep) and len(np.unique(d["y"][keep])) > 1:
                a2, lo2, hi2 = auc_ci(d["y"][keep], d["best"][keep])
                ext_rows[-1]["AUC excl. borderline (sim 0.90-0.95)"] = \
                    f"{a2:.3f} ({lo2:.3f}-{hi2:.3f}), n={int(keep.sum())}"
            pd.DataFrame({"subject_id": d["subject_id"], "MGMT_value": d["y"], "pred": d["best"]}
                         ).to_csv(os.path.join(OUT_DIR, f"external_preds_{site}.csv"), index=False)
        ext_tbl = pd.DataFrame(ext_rows)
        ext_tbl.to_csv(os.path.join(OUT_DIR, "external_table.csv"), index=False)
        print("\nLEAVE-ONE-SITE-OUT (trained without the site, duplicates removed)")
        print(ext_tbl.to_string(index=False))

    with open(os.path.join(OUT_DIR, "results.json"), "w") as fh:
        json.dump({"cohort": cohort, "main": tbl.to_dict("records"),
                   "external": ext_tbl.to_dict("records") if ext_tbl is not None else []},
                  fh, indent=2, default=str)
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        cols = ["Model", "AUC (95% CI)", "AUC strict", "MCC"] + \
               [c for c in ("dAUC vs paper", "DeLong p vs paper") if c in tbl.columns]
        t = tbl[cols]
        fig, ax = plt.subplots(figsize=(13, 1.6 + 0.45 * len(t))); ax.axis("off")
        ax.set_title(f"Multi-cohort training, evaluated on the paper's {len(y)} patients",
                     fontsize=11, fontweight="bold", loc="left")
        tb = ax.table(cellText=t.values, colLabels=t.columns, cellLoc="center", loc="center")
        tb.auto_set_font_size(False); tb.set_fontsize(8); tb.scale(1, 1.5)
        tb.auto_set_column_width(col=list(range(len(cols))))
        for (r, c), cell in tb.get_celld().items():
            if r == 0: cell.set_facecolor("#1f3a5f"); cell.set_text_props(color="w", fontweight="bold")
            elif "primary" in str(t.iloc[r - 1, 0]): cell.set_facecolor("#e8f0fe")
        fig.savefig(os.path.join(OUT_DIR, "results_table.png"), dpi=200, bbox_inches="tight")
    except Exception as e:
        print("(table image skipped:", e, ")")
    print("\nsaved to", os.path.abspath(OUT_DIR))


# ------------------------------- ablation --------------------------------------
def ablation_report(df, y, fold42, groups, abl):
    """Remove one component of the proposed B0+B2 ensemble at a time. Every row is
    an ensemble of the same backbones (except '− ensemble'), compared with the full
    model on the same 574 patients (DeLong)."""
    from sklearn.metrics import roc_auc_score
    find = lambda kind, bb: next((g for g in groups.values()
                                  if g.get("kind") == kind and g.get("backbone") == bb), None)
    merged = {bb: find("cv", bb) for bb in BACKBONES}
    if any(g is None for g in merged.values()):
        print("ablation: proposed model incomplete -> skipped"); return
    ens = lambda arrs: np.mean([rankise(a) for a in arrs], 0)
    mean_mod = lambda pm: np.mean([rankise(pm[m]) for m in MODALITIES], 0)
    both = lambda f: (ens([f(b, "best") for b in BACKBONES]), ens([f(b, "last") for b in BACKBONES]))
    rows_def = [("Proposed (full)", *both(lambda b, w: merged[b][w]))]
    ctrl = {bb: find("control", bb) for bb in BACKBONES}
    if all(ctrl.values()):
        rows_def.append(("− Multi-cohort data (BraTS only)", *both(lambda b, w: ctrl[b][w])))
    if len(BACKBONES) > 1:
        b2 = merged[BACKBONES[0]]
        rows_def.append((f"− Ensemble (EfficientNet-{short(BACKBONES[0]).upper()} only)",
                         b2["best"], b2["last"]))
    rows_def.append(("− Skill weighting (plain mean of modalities)",
                     *both(lambda b, w: mean_mod(merged[b]["pm_" + w]))))
    if "FLAIR" in MODALITIES:
        rows_def.append(("− Multimodal fusion (FLAIR only)",
                         *both(lambda b, w: merged[b]["pm_" + w]["FLAIR"])))
    for v in ABLATIONS:
        if v in abl and all(bb in abl[v] for bb in BACKBONES):
            rows_def.append((ABLATION_LABELS[v], *both(lambda b, w: abl[v][b][w])))

    full, full_last = rows_def[0][1], rows_def[0][2]
    a_full, s_full = roc_auc_score(y, full), roc_auc_score(y, full_last)
    rows = []
    for k, (name, p, pl) in enumerate(rows_def):
        a, lo, hi = auc_ci(y, p); s = roc_auc_score(y, pl)
        row = {"Variant": name, "AUC (95% CI)": f"{a:.3f} ({lo:.3f}-{hi:.3f})",
               "dAUC": "-" if k == 0 else f"{a - a_full:+.3f}",
               "DeLong p vs full": "-" if k == 0 else f"{delong(y, full, p)[1]:.4f}",
               "AUC strict": f"{s:.3f}", "dAUC strict": "-" if k == 0 else f"{s - s_full:+.3f}",
               "MCC": round(mcc_youden(y, p), 3)}
        for f in range(N_FOLDS):
            m = fold42 == f; row[f"Fold {f+1}"] = round(roc_auc_score(y[m], p[m]), 3)
        rows.append(row)
        slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")[:60]
        pd.DataFrame({"BraTS21ID": df.BraTS21ID.values, "MGMT_value": y, "pred": p, "pred_strict": pl,
                      "fold": fold42}).to_csv(os.path.join(OUT_DIR, f"oof_ablation_{slug}.csv"), index=False)
    tbl = pd.DataFrame(rows)
    tbl.to_csv(os.path.join(OUT_DIR, "ablation_table.csv"), index=False)
    print("\n" + "=" * 110 + "\nABLATION of the proposed model (one component removed at a time)\n" + "=" * 110)
    print(tbl.to_string(index=False))
    print("dAUC < 0 means the removed component helps. 'strict' = final epoch (no selection on the test fold).")


# ------------------------------- Grad-CAM --------------------------------------
def gradcam_figure(df, y, brats_cache):
    """Grad-CAM heat-maps of the saved B2 models on held-out fold-1 patients
    (the 4 most confident correct methylated and 4 unmethylated cases)."""
    import torch, cv2
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from sklearn.model_selection import StratifiedKFold
    mods = [m for m in GRADCAM_MODS if m in MODALITIES]
    paths = {m: os.path.join(WEIGHT_DIR, job_name("cam", GRADCAM_BACKBONE, m, SEEDS[0], 0) + ".pt")
             for m in mods}
    if not mods or not all(os.path.isfile(p) for p in paths.values()):
        print("Grad-CAM: saved model weights not found -> skipped"); return
    ids = df.BraTS21ID.astype(str).values
    _, va = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEEDS[0]).split(ids, y))[0]
    dev = torch.device("cuda:0" if DEVICE == "cuda" and torch.cuda.is_available() else "cpu")

    def load_model(p):
        m = make_model(GRADCAM_BACKBONE, False)
        m.load_state_dict(torch.load(p, map_location="cpu")); return m.to(dev).eval()

    def cam(model, arr):
        x = torch.from_numpy(arr).permute(2, 0, 1)[None].float().to(dev)
        x = (x / 255.0 - 0.5) / 0.5
        feat = model.forward_features(x); feat.retain_grad()
        logit = model.forward_head(feat).reshape(())
        model.zero_grad(set_to_none=True); logit.backward()
        w = feat.grad.mean(dim=(2, 3), keepdim=True)
        c = torch.relu((w * feat).sum(1))[0].detach().cpu().numpy()
        c = cv2.resize(c, (arr.shape[1], arr.shape[0])); c = c / (c.max() + 1e-8)
        return c, float(torch.sigmoid(logit).item())

    models = {m: load_model(p) for m, p in paths.items()}
    def load_arr(m, i):
        a = np.load(os.path.join(brats_cache, f"{m}_{i}.npy"))
        return np.ascontiguousarray(np.repeat(a[:, :, 1:2], 3, axis=2)) if SINGLE_SLICE else a
    arrs = {m: {i: load_arr(m, i) for i in ids[va]} for m in mods}
    yv = dict(zip(ids[va], y[va]))
    probs = {i: cam(models[mods[0]], arrs[mods[0]][i])[1] for i in ids[va]}
    pos = sorted([i for i in ids[va] if yv[i] == 1 and probs[i] > 0.5], key=lambda i: -probs[i])[:4]
    neg = sorted([i for i in ids[va] if yv[i] == 0 and probs[i] < 0.5], key=lambda i: probs[i])[:4]
    pick = pos + neg
    if not pick:
        print("Grad-CAM: no confidently correct cases -> skipped"); return
    fig, axes = plt.subplots(len(pick), 2 * len(mods), figsize=(5.2 * len(mods), 2.7 * len(pick)), squeeze=False)
    for r, i in enumerate(pick):
        for k, m in enumerate(mods):
            c, p = cam(models[m], arrs[m][i]); img = arrs[m][i][:, :, 1]
            ax = axes[r, 2 * k]; ax.imshow(img, cmap="gray"); ax.axis("off")
            ax.set_title(f"{m} | BraTS21 {i} | {'methylated' if yv[i] else 'unmethylated'}", fontsize=8)
            ax = axes[r, 2 * k + 1]; ax.imshow(img, cmap="gray"); ax.imshow(c, cmap="jet", alpha=0.4)
            ax.axis("off"); ax.set_title(f"Grad-CAM | p(methylated) = {p:.2f}", fontsize=8)
    fig.suptitle(f"Grad-CAM, {GRADCAM_BACKBONE} (held-out fold 1; most confident correct cases)", fontsize=10)
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(OUT_DIR, f"fig_gradcam.{ext}"), dpi=200, bbox_inches="tight")
    print("saved fig_gradcam.png / .pdf")


# ---------------------------------- main --------------------------------------
def preflight():
    import torch
    if DEVICE != "cuda": print(f"PREFLIGHT: running on {DEVICE} (test mode)"); return
    if not torch.cuda.is_available():
        sys.exit("PREFLIGHT: no GPU. Settings -> Accelerator -> GPU T4 x2, then re-run.")
    n = torch.cuda.device_count()
    for i in range(n):
        cap = torch.cuda.get_device_capability(i)
        print(f"PREFLIGHT: GPU{i} {torch.cuda.get_device_name(i)} sm_{cap[0]}{cap[1]}")
    if n < 2:
        print("PREFLIGHT WARNING: only one GPU visible -- choose 'GPU T4 x2' to halve the run time.")



def print_run_mode():
    """Say whether this is a background commit (safe) or an interactive session."""
    mode = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
    bar = "#" * 78
    if mode == "Batch":
        msg = ("RUN MODE: BACKGROUND COMMIT (Save & Run All)\n"
               "Safe to close the laptop. Output is kept under this version's Output tab.")
    elif mode == "Interactive":
        msg = ("RUN MODE: INTERACTIVE SESSION\n"
               "Output is LOST when the session stops. Use this for the quick test only;\n"
               "for the full run use Save Version -> Save & Run All (Commit).")
    else:
        msg = "RUN MODE: not on Kaggle (local run)"
    print(f"{bar}\n{msg}\n{bar}\n", flush=True)


def main():
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedKFold
    t_start = time.time()
    print_run_mode()
    print(f"SCRIPT VERSION: {SCRIPT_VERSION}\n", flush=True)
    os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CACHE_ROOT, exist_ok=True)
    print("QUICK_TEST" if QUICK_TEST else "FULL RUN", "| epochs", EPOCHS, "| seeds", SEEDS,
          "| backbones", BACKBONES)
    list_inputs()

    # ---- evaluation cohort: exactly the paper's 574 patients ----------------
    lab = find_files("train_labels.csv")
    assert lab, "train_labels.csv not found -- add the RSNA-MICCAI competition as an input."
    train_df = pd.read_csv(lab[0])
    train_df = train_df[~train_df.BraTS21ID.isin(BAD_IDS)].reset_index(drop=True)
    source = find_brats_source()
    brats_cache, covered = build_brats_cache(set(train_df.BraTS21ID.astype(int)), source)
    df = train_df[train_df.BraTS21ID.astype(int).isin(covered)].reset_index(drop=True)
    y = df.MGMT_value.values.astype(int)
    print(f"evaluation cohort: {len(df)} patients, positive rate {y.mean():.3f}")
    if len(df) != 574 and not QUICK_TEST and ON_KAGGLE:
        print(f"WARNING: expected 574 patients (paper), got {len(df)}.")

    # ---- extra cohorts --------------------------------------------------------
    axcodes = brats_axcodes(source) if source[0] else ("L", "P", "S")
    print("BraTS orientation:", axcodes)
    sites, caches, cohort, dedup = {}, {"BRATS": brats_cache}, {"BRATS_eval": int(len(df))}, []
    site_meta = {}
    for site in EXTRA_SITES:
        cache, meta = build_site_cache(site, axcodes)
        if meta is None or meta.empty: continue
        site_meta[site] = meta
        rep = dedup_audit(site, meta, cache, brats_cache, [str(i) for i in df.BraTS21ID],
                          {str(i): int(v) for i, v in zip(df.BraTS21ID, y)})
        dedup.append(rep)
        keep = meta[~meta.idx.isin(rep.loc[rep.excluded, "idx"])]
        sites[site] = {"ids": [str(i) for i in keep.idx], "y": [int(v) for v in keep.MGMT_value],
                       "subject_id": list(keep.subject_id)}
        caches[site] = cache
        cohort[site] = {"cached": int(len(meta)), "excluded_duplicates": int(rep.excluded.sum()),
                        "used": int(len(keep)), "pos_rate": round(float(keep.MGMT_value.mean()), 3)}
    if dedup:
        pd.concat(dedup).to_csv(os.path.join(OUT_DIR, "dedup_report.csv"), index=False)
    if not sites:
        print("\nWARNING: no extra cohort found -- only the BraTS-only control can run.")
    print("cohort summary:", json.dumps(cohort))
    n_train = int(len(df) * (N_FOLDS - 1) / N_FOLDS) + sum(len(s["ids"]) for s in sites.values())
    print(f"training patients per fold: ~{n_train} (paper: ~{int(len(df)*0.8)})")

    # ---- ablation image caches (whole slice / centre slice) -------------------
    caches_all = {"main": caches}
    abl_ok = []
    if RUN_ABLATION and sites:
        for v in ABLATIONS:
            if v in CACHE_VARIANTS:
                bdir, _ = build_brats_cache(set(df.BraTS21ID.astype(int)), source, v)
                vc = {"BRATS": bdir}
                for site in sites:
                    vc[site] = build_site_variant_cache(site, axcodes, v, site_meta[site])
                if cache_complete(bdir, [str(i) for i in df.BraTS21ID]) and \
                        all(cache_complete(vc[s], sites[s]["ids"]) for s in sites):
                    caches_all[v] = vc
                else:
                    print(f"WARNING: '{v}' images incomplete -> that ablation row is skipped"); continue
            abl_ok.append(v)

    # ---- job list -------------------------------------------------------------
    jobs = []
    def add_jobs(kind, backbone, train_sites, test_site=None, variant="main"):
        for mod in MODALITIES:
            for seed in SEEDS:
                for fold in range(N_FOLDS):
                    jobs.append({"name": job_name(kind, backbone, mod, seed, fold),
                                 "backbone": backbone, "modality": mod, "seed": seed, "fold": fold,
                                 "epochs": EPOCHS, "train_sites": train_sites, "test_site": test_site,
                                 "variant": variant})
    if sites:
        for bb in BACKBONES: add_jobs("cv", bb, list(sites))
        if RUN_BASELINES:
            for bb in BASELINE_BACKBONES: add_jobs("base", bb, list(sites))
        if RUN_EXTERNAL:
            for held in sites:
                for bb in EXTERNAL_BACKBONES:
                    add_jobs(f"ext{held}", bb, [s for s in sites if s != held], held)
        if RUN_GRADCAM:      # short: 1 seed x fold 1, weights kept for the heat-maps
            for mod in GRADCAM_MODS:
                jobs.append({"name": job_name("cam", GRADCAM_BACKBONE, mod, SEEDS[0], 0),
                             "backbone": GRADCAM_BACKBONE, "modality": mod, "seed": SEEDS[0],
                             "fold": 0, "epochs": EPOCHS, "train_sites": list(sites),
                             "test_site": None, "variant": "main", "save_weights": True})
        for v in abl_ok:
            for bb in BACKBONES: add_jobs(f"abl{v}", bb, list(sites), variant=v)
    if RUN_CONTROL or not sites:
        add_jobs("control", "efficientnet_b0", [])
    if RUN_CONTROL_B2 and sites:
        add_jobs("control", "efficientnet_b2", [])

    restore_jobs_from_inputs()
    preflight()
    if PRETRAINED:   # download weights once, before the workers start
        import timm
        for bb in set(j["backbone"] for j in jobs):
            if bb != "simple_cnn": timm.create_model(bb, pretrained=True)
    cfg = {"brats_ids": [str(i) for i in df.BraTS21ID], "brats_y": [int(v) for v in y],
           "sites": sites, "caches": caches_all, "seeds": SEEDS,
           "deadline": t_start + TIME_BUDGET_H * 3600}
    run_jobs(jobs, cfg)

    # ---- assemble + report ----------------------------------------------------
    fold42 = np.empty(len(y), int)
    for f, (_, va) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEEDS[0]).split(y, y)):
        fold42[va] = f
    groups, missing = {}, []
    group_defs = [("cv", bb, f"Merged {short(bb).upper()} (BraTS + {'+'.join(sites)})") for bb in BACKBONES] \
        if sites else []
    if RUN_BASELINES and sites:
        group_defs += [("base", bb, f"Baseline {BASELINE_LABELS.get(bb, bb)} (BraTS + {'+'.join(sites)})")
                       for bb in BASELINE_BACKBONES]
    if RUN_CONTROL or not sites:
        group_defs.append(("control", "efficientnet_b0", "Control B0 (BraTS only, this code)"))
    if RUN_CONTROL_B2 and sites:
        group_defs.append(("control", "efficientnet_b2", "Control B2 (BraTS only, this code)"))
    for kind, bb, label in group_defs:
        best, last = assemble_cv(kind, bb, len(y), "best_va"), assemble_cv(kind, bb, len(y), "last_va")
        if best is None: missing.append(label); continue
        groups[label] = {"best": skill_weighted(best, y), "last": skill_weighted(last, y),
                         "pm_best": best, "pm_last": last, "backbone": bb, "kind": kind,
                         "per_mod": {m: round(float(roc_auc_score(y, best[m])), 3) for m in MODALITIES}}
        print(f"{label}: per-modality AUC {groups[label]['per_mod']}")
    abl = {}
    for v in abl_ok:
        for bb in BACKBONES:
            b, l = assemble_cv(f"abl{v}", bb, len(y), "best_va"), assemble_cv(f"abl{v}", bb, len(y), "last_va")
            if b is None: missing.append(f"ablation {v} {short(bb).upper()}"); continue
            abl.setdefault(v, {})[bb] = {"best": skill_weighted(b, y), "last": skill_weighted(l, y)}
    external = {}
    if RUN_EXTERNAL:
        for held, s in sites.items():
            # one prediction per backbone, then rank-averaged over backbones and MRI types
            bests = [assemble_ext(held, bb, len(s["y"]), "best_te") for bb in EXTERNAL_BACKBONES]
            lasts = [assemble_ext(held, bb, len(s["y"]), "last_te") for bb in EXTERNAL_BACKBONES]
            if any(b is None for b in bests): missing.append(f"external {held}"); continue
            rep = next((r for r in dedup if len(r) and r.site.iloc[0] == held), None)
            near = set() if rep is None else set(rep.loc[~rep.excluded & (rep.max_cosine > 0.90), "subject_id"])
            external[held] = {"y": np.array(s["y"]), "subject_id": s["subject_id"],
                              "per_mod": {m: np.mean([rankise(b[m]) for b in bests], 0) for m in MODALITIES},
                              "borderline": near,
                              "best": np.mean([rankise(b[m]) for b in bests for m in MODALITIES], 0),
                              "last": np.mean([rankise(l[m]) for l in lasts for m in MODALITIES], 0)}
    if missing:
        print("\nINCOMPLETE (re-run with this output attached as input to finish):", missing)
    prior = load_prior(df)
    if groups or prior:
        report(df, y, fold42, groups, prior, external, cohort)
    for step, fn in (("ablation", lambda: ablation_report(df, y, fold42, groups, abl)),
                     ("Grad-CAM", lambda: gradcam_figure(df, y, caches["BRATS"]) if RUN_GRADCAM else None)):
        try:
            fn()
        except Exception:
            print(f"({step} step failed -- other results are unaffected)\n{traceback.format_exc()}")
    print(f"\nDONE in {(time.time()-t_start)/3600:.2f} h")


if __name__ == "__main__":
    if len(sys.argv) >= 4 and sys.argv[1] == "--worker":
        _worker_cli(int(sys.argv[2]), sys.argv[3])     # one GPU process
    else:
        main()
''',
    'paper_figures.py': r'''#!/usr/bin/env python3
# =============================================================================
# paper_figures.py  --  all paper figures + statistics from the saved results
# -----------------------------------------------------------------------------
# Reads the output folder of kaggle_merged_training.py (no GPU, no training):
#     !python paper_figures.py /kaggle/working/outputs_merged
# Writes <output folder>/figures/ :
#   fig1_roc_comparison        ROC: proposed vs all baselines vs paper model
#   fig2_pr_comparison         precision-recall curves
#   fig3_auc_forest            AUC with 95% CI for every model
#   fig4_bootstrap             bootstrap AUC + paired dAUC vs the paper model
#   fig5_perfold_auc           per-fold AUC: proposed vs paper vs BraTS-only control
#   fig6_confusion_matrix      proposed model at p = 0.5 (calibrated)
#   fig7_external_roc          leave-one-site-out ROC (UPenn, UCSF)
#   fig8_calibration           reliability diagram + Brier score
#   fig9_decision_curve        net benefit (decision curve analysis)
#   fig10_selective            accuracy vs coverage (confidence-based referral)
#   fig11_ablation             ablation dAUC (if ablation_table.csv exists)
#   table_comparison_holm.csv  DeLong vs proposed, Holm-adjusted across baselines
#   table_clinical_metrics.csv sensitivity/specificity/PPV/NPV/F1/MCC/Brier/ECE
#   table_efficiency.csv       parameters per network and per patient
#   table_selective.csv        accuracy at 100/50/30/20 % coverage
#   paper_numbers.txt          every key number in one place
# Each figure is saved as .png (200 dpi) and .pdf. A failing figure is reported
# and skipped; the others are still produced.
# =============================================================================
import os, re, sys, glob, json, traceback, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, confusion_matrix, matthews_corrcoef, f1_score)
from sklearn.linear_model import LogisticRegression
from scipy.stats import norm, rankdata

OUT = sys.argv[1] if len(sys.argv) > 1 else "/kaggle/working/outputs_merged"
FIG = os.path.join(OUT, "figures"); os.makedirs(FIG, exist_ok=True)
SEED, B = 42, 2000
PRIMARY_COLOR, PAPER_COLOR = "#1f3a8a", "#6b7280"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
numbers = []          # lines for paper_numbers.txt


def note(line):
    print(line); numbers.append(line)


def slug(name):
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")[:60]


def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{name}.{ext}"), dpi=200, bbox_inches="tight")
    plt.close(fig); print("  saved", name)


# ---------------------------------------------------------------- statistics
def delong(y, p1, p2):
    y = np.asarray(y).astype(int); pos, neg = y == 1, y == 0
    def comps(p):
        d = p[pos][:, None] - p[neg][None, :]
        psi = (d > 0) + 0.5 * (d == 0)
        return psi.mean(1), psi.mean(0), psi.mean()
    a10, a01, a = comps(np.asarray(p1)); b10, b01, b = comps(np.asarray(p2))
    S = np.cov(np.vstack([a10, b10])) / pos.sum() + np.cov(np.vstack([a01, b01])) / neg.sum()
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    if var <= 0: return 0.0, 1.0
    z = (a - b) / np.sqrt(var); return float(z), float(2 * norm.sf(abs(z)))


def boot_auc(y, p, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y); out = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        if len(np.unique(y[i])) > 1: out.append(roc_auc_score(y[i], p[i]))
    return np.array(out)


def ci(y, p):
    b = boot_auc(y, p); return roc_auc_score(y, p), np.percentile(b, 2.5), np.percentile(b, 97.5)


def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p); adj = np.empty(m); run = 0.0
    for k, i in enumerate(order):
        run = max(run, min(1.0, (m - k) * p[i])); adj[i] = run
    return adj


def calibrate(score, y, fold):
    """Cross-fitted Platt scaling: each fold is calibrated on the other folds only."""
    prob = np.zeros(len(y))
    for f in np.unique(fold):
        tr, te = fold != f, fold == f
        lr = LogisticRegression(C=1e6).fit(score[tr, None], y[tr])
        prob[te] = lr.predict_proba(score[te, None])[:, 1]
    return prob


def metrics_at(y, prob, thr):
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"threshold": round(float(thr), 3), "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
            "PPV": tp / max(tp + fp, 1), "NPV": tn / max(tn + fn, 1), "accuracy": (tp + tn) / len(y),
            "F1": f1_score(y, pred), "MCC": matthews_corrcoef(y, pred), "TP": tp, "FP": fp, "TN": tn, "FN": fn}


def ece(y, prob, bins=10):
    edges = np.quantile(prob, np.linspace(0, 1, bins + 1)); idx = np.clip(np.searchsorted(edges, prob, "right") - 1, 0, bins - 1)
    return float(sum(abs(prob[idx == b].mean() - y[idx == b].mean()) * (idx == b).mean()
                     for b in range(bins) if (idx == b).any()))


# ---------------------------------------------------------------- load results
res = pd.read_csv(os.path.join(OUT, "results_table.csv"))
models = {}
for name in res.Model:
    f = os.path.join(OUT, f"oof_merged_{slug(name)}.csv")
    if os.path.isfile(f): models[name] = pd.read_csv(f)
any_df = next(iter(models.values()))
y = any_df.MGMT_value.values.astype(int); fold = any_df.fold.values.astype(int)
P = {n: d.pred.values.astype(float) for n, d in models.items()}
primary = next(n for n in P if "primary" in n)
paper = next((n for n in P if n.startswith("Paper headline")), None)
control = next((n for n in P if n.startswith("Control B0")), None)
baselines = [n for n in P if n.startswith("Baseline")]
merged_single = [n for n in P if n.startswith("Merged B")]
short = lambda n: (n.replace("  <- primary", "").replace(" (BraTS + UPENN+UCSF)", "")
                   .replace("Baseline ", "").replace(" (BraTS only)", " (BraTS only)"))
PRIMARY_LABEL = "Proposed: multi-cohort EfficientNet B0+B2"
label = lambda n: PRIMARY_LABEL if n == primary else ("Paper model (B0, BraTS only)" if n == paper else short(n))
note(f"Patients: {len(y)} (methylated {y.sum()}, rate {y.mean():.3f}); models found: {len(P)}")

palette = plt.cm.tab10.colors
order = [primary] + baselines + ([paper] if paper else [])
colors = {n: palette[(i + 1) % 10] for i, n in enumerate(baselines)}
colors[primary] = PRIMARY_COLOR
if paper: colors[paper] = PAPER_COLOR


def run(name, fn):
    try:
        fn()
    except Exception:
        print(f"  FIGURE FAILED: {name}\n{traceback.format_exc()}")


# ---------------------------------------------------------------- fig 1 / 2
def fig_roc_pr():
    fig, ax = plt.subplots(figsize=(5.6, 5))
    for n in order:
        fpr, tpr, _ = roc_curve(y, P[n]); a = roc_auc_score(y, P[n])
        ax.plot(fpr, tpr, color=colors[n], lw=2.6 if n == primary else 1.3,
                ls="--" if n == paper else "-", label=f"{label(n)} (AUC {a:.3f})")
    ax.plot([0, 1], [0, 1], ":", color="#9ca3af", lw=1)
    ax.set_xlabel("1 - specificity"); ax.set_ylabel("Sensitivity")
    ax.set_title("ROC curves (same 574 patients, 5-fold cross-validation)")
    ax.legend(fontsize=7, loc="lower right", frameon=False); save(fig, "fig1_roc_comparison")

    fig, ax = plt.subplots(figsize=(5.6, 5))
    for n in order:
        pr, rc, _ = precision_recall_curve(y, P[n]); ap = average_precision_score(y, P[n])
        ax.plot(rc, pr, color=colors[n], lw=2.6 if n == primary else 1.3,
                ls="--" if n == paper else "-", label=f"{label(n)} (AP {ap:.3f})")
    ax.axhline(y.mean(), ls=":", color="#9ca3af", lw=1, label=f"Prevalence ({y.mean():.2f})")
    ax.set_xlabel("Recall (sensitivity)"); ax.set_ylabel("Precision (PPV)")
    ax.set_title("Precision-recall curves"); ax.legend(fontsize=7, loc="lower left", frameon=False)
    save(fig, "fig2_pr_comparison")


# ---------------------------------------------------------------- fig 3 + Holm table
def fig_forest_and_holm():
    rows, names = [], [primary] + baselines + [n for n in (control, paper) if n]
    for n in names:
        a, lo, hi = ci(y, P[n]); rows.append((n, a, lo, hi))
    fig, ax = plt.subplots(figsize=(6.2, 0.42 * len(rows) + 1))
    for k, (n, a, lo, hi) in enumerate(rows[::-1]):
        c = colors.get(n, "#374151")
        ax.errorbar(a, k, xerr=[[a - lo], [hi - a]], fmt="o", color=c, ms=6 if n == primary else 4, capsize=3,
                    lw=2 if n == primary else 1.2)
        ax.text(hi + 0.005, k, f"{a:.3f} ({lo:.3f}-{hi:.3f})", va="center", fontsize=7)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels([label(n) for n, *_ in rows[::-1]], fontsize=8)
    ax.axvline(0.5, ls=":", color="#9ca3af"); ax.set_xlim(0.5, 0.86)
    ax.set_xlabel("AUC (95% bootstrap CI)"); ax.set_title("Discrimination of all models")
    save(fig, "fig3_auc_forest")

    comp = baselines + [n for n in (control, paper) if n]
    HOLM_COL = f"Holm-adjusted p ({len(baselines)} baselines)"
    zp = [delong(y, P[primary], P[n]) for n in comp]
    adj = holm([p for _, p in zp[:len(baselines)]]) if baselines else []
    out = []
    for k, n in enumerate(comp):
        a, lo, hi = ci(y, P[n]); z, p = zp[k]
        h = adj[k] if k < len(baselines) else np.nan
        out.append({"Model": label(n), "AUC (95% CI)": f"{a:.3f} ({lo:.3f}-{hi:.3f})",
                    "Proposed - model": f"{roc_auc_score(y, P[primary]) - a:+.3f}", "DeLong z": round(z, 2),
                    "DeLong p": round(p, 4),
                    HOLM_COL: "-" if np.isnan(h) else round(float(h), 4),
                    "Significant after Holm (0.05)": "-" if np.isnan(h) else ("yes" if h < 0.05 else "no")})
    t = pd.DataFrame(out); t.to_csv(os.path.join(FIG, "table_comparison_holm.csv"), index=False)
    print(t.to_string(index=False))
    for r in out: note(f"vs {r['Model']}: dAUC {r['Proposed - model']}, DeLong p {r['DeLong p']}, Holm {r[HOLM_COL]}")


# ---------------------------------------------------------------- fig 4
def fig_bootstrap():
    rng = np.random.default_rng(SEED); n = len(y); a_b, d_b = [], []
    for _ in range(B):
        i = rng.integers(0, n, n)
        if len(np.unique(y[i])) < 2: continue
        a = roc_auc_score(y[i], P[primary][i]); a_b.append(a)
        if paper: d_b.append(a - roc_auc_score(y[i], P[paper][i]))
    a_b, d_b = np.array(a_b), np.array(d_b)
    fig, axes = plt.subplots(1, 2 if paper else 1, figsize=(10 if paper else 5.2, 3.8), squeeze=False)
    ax = axes[0, 0]; lo, hi = np.percentile(a_b, [2.5, 97.5])
    ax.hist(a_b, bins=40, color=PRIMARY_COLOR, alpha=0.8)
    ax.axvspan(lo, hi, color=PRIMARY_COLOR, alpha=0.08)
    if paper: ax.axvline(roc_auc_score(y, P[paper]), color=PAPER_COLOR, ls="--", label="Paper model AUC")
    if control: ax.axvline(roc_auc_score(y, P[control]), color="#b45309", ls=":", label="BraTS-only control")
    ax.set_title(f"Bootstrap AUC of the proposed model (B={len(a_b)})\n95% CI {lo:.3f}-{hi:.3f}")
    ax.set_xlabel("AUC"); ax.legend(fontsize=7, frameon=False)
    note(f"Proposed AUC bootstrap 95% CI {lo:.3f}-{hi:.3f}")
    if paper:
        ax = axes[0, 1]; lo, hi = np.percentile(d_b, [2.5, 97.5]); frac = (d_b > 0).mean()
        ax.hist(d_b, bins=40, color="#047857", alpha=0.8); ax.axvline(0, color="k", lw=1)
        ax.set_title(f"Paired bootstrap: proposed - paper model\n"
                     f"dAUC 95% CI {lo:+.3f} to {hi:+.3f}; P(dAUC > 0) = {frac:.3f}")
        ax.set_xlabel("dAUC")
        note(f"Paired dAUC vs paper model: 95% CI {lo:+.3f} to {hi:+.3f}, P(>0) = {frac:.3f}")
    save(fig, "fig4_bootstrap")


# ---------------------------------------------------------------- fig 5
def fig_perfold():
    names = [n for n in (primary, control, paper) if n]
    fig, ax = plt.subplots(figsize=(6.2, 3.6)); w = 0.8 / len(names); folds = sorted(np.unique(fold))
    for k, n in enumerate(names):
        vals = [roc_auc_score(y[fold == f], P[n][fold == f]) for f in folds]
        ax.bar(np.arange(len(folds)) + k * w, vals, w, label=f"{label(n)} (mean {np.mean(vals):.3f})",
               color=colors.get(n, "#b45309"))
    ax.set_xticks(np.arange(len(folds)) + w * (len(names) - 1) / 2)
    ax.set_xticklabels([f"Fold {f+1}" for f in folds]); ax.set_ylim(0.5, 0.85); ax.set_ylabel("AUC")
    ax.set_title("Per-fold AUC on the held-out folds"); ax.legend(fontsize=7, frameon=False, loc="upper left")
    save(fig, "fig5_perfold_auc")


# ---------------------------------------------------------------- calibrated probabilities
PROB = None


def calibrated():
    global PROB
    if PROB is None: PROB = calibrate(P[primary], y, fold)
    return PROB


# ---------------------------------------------------------------- fig 6 + clinical table
def fig_confusion_and_clinical():
    prob = calibrated()
    m = metrics_at(y, prob, 0.5)
    fpr, tpr, thr = roc_curve(y, prob); spec = 1 - fpr
    k = np.where(spec >= 0.90)[0][-1]; m90 = metrics_at(y, prob, thr[k])
    brier = float(np.mean((prob - y) ** 2)); e = ece(y, prob)
    rows = [dict(operating_point="balanced (p >= 0.5)", **m), dict(operating_point="high specificity (>= 0.90)", **m90)]
    t = pd.DataFrame(rows)
    for c in ("sensitivity", "specificity", "PPV", "NPV", "accuracy", "F1", "MCC"): t[c] = t[c].round(3)
    t["Brier"] = round(brier, 3); t["ECE"] = round(e, 3)
    t.to_csv(os.path.join(FIG, "table_clinical_metrics.csv"), index=False); print(t.to_string(index=False))
    note(f"At p>=0.5: sens {m['sensitivity']:.3f}, spec {m['specificity']:.3f}, PPV {m['PPV']:.3f}, "
         f"NPV {m['NPV']:.3f}, acc {m['accuracy']:.3f}, MCC {m['MCC']:.3f}")
    note(f"At spec>=0.90: sens {m90['sensitivity']:.3f}, spec {m90['specificity']:.3f}, PPV {m90['PPV']:.3f}")
    note(f"Calibration: Brier {brier:.3f}, ECE {e:.3f}")
    cm = np.array([[m["TN"], m["FP"]], [m["FN"], m["TP"]]])
    fig, ax = plt.subplots(figsize=(3.8, 3.4)); ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Unmethylated", "Methylated"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Unmethylated", "Methylated"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"Proposed model, p >= 0.5\nsens {m['sensitivity']:.2f} | spec {m['specificity']:.2f} | "
                 f"acc {m['accuracy']:.2f}", fontsize=8)
    save(fig, "fig6_confusion_matrix")


# ---------------------------------------------------------------- fig 7
def fig_external():
    files = sorted(glob.glob(os.path.join(OUT, "external_preds_*.csv")))
    if not files: print("  no external predictions -> fig7 skipped"); return
    dd = os.path.join(OUT, "dedup_report.csv")
    dedup = pd.read_csv(dd) if os.path.isfile(dd) else None
    fig, ax = plt.subplots(figsize=(5, 4.6))
    for f, c in zip(files, ["#1f3a8a", "#b91c1c", "#047857"]):
        site = os.path.basename(f).replace("external_preds_", "").replace(".csv", "")
        d = pd.read_csv(f); yy, pp = d.MGMT_value.values.astype(int), d.pred.values
        a, lo, hi = ci(yy, pp); fpr, tpr, _ = roc_curve(yy, pp)
        ax.plot(fpr, tpr, color=c, lw=2, label=f"{site} (n={len(yy)}) AUC {a:.3f} ({lo:.3f}-{hi:.3f})")
        note(f"External {site}: n={len(yy)}, AUC {a:.3f} ({lo:.3f}-{hi:.3f})")
        if dedup is not None:
            near = set(dedup.loc[(dedup.site == site) & (~dedup.excluded.astype(bool)) & (dedup.max_cosine > 0.90),
                                 "subject_id"])
            keep = ~d.subject_id.isin(near).values
            if 0 < (~keep).sum() and len(np.unique(yy[keep])) > 1:
                a2, lo2, hi2 = ci(yy[keep], pp[keep]); fpr, tpr, _ = roc_curve(yy[keep], pp[keep])
                ax.plot(fpr, tpr, color=c, lw=1.4, ls="--",
                        label=f"{site} excl. {int((~keep).sum())} borderline (n={int(keep.sum())}) AUC {a2:.3f}")
                note(f"External {site} excluding borderline (sim 0.90-0.95): n={int(keep.sum())}, "
                     f"AUC {a2:.3f} ({lo2:.3f}-{hi2:.3f})")
    ax.plot([0, 1], [0, 1], ":", color="#9ca3af")
    ax.set_xlabel("1 - specificity"); ax.set_ylabel("Sensitivity")
    ax.set_title("Leave-one-site-out external validation\n(model never trained on the held-out site)")
    ax.legend(fontsize=7, loc="lower right", frameon=False); save(fig, "fig7_external_roc")


# ---------------------------------------------------------------- fig 8
def fig_calibration():
    prob = calibrated(); bins = 10
    edges = np.quantile(prob, np.linspace(0, 1, bins + 1))
    idx = np.clip(np.searchsorted(edges, prob, "right") - 1, 0, bins - 1)
    mp = [prob[idx == b].mean() for b in range(bins) if (idx == b).any()]
    fo = [y[idx == b].mean() for b in range(bins) if (idx == b).any()]
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(4.6, 5.4), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
    ax.plot([0, 1], [0, 1], ":", color="#9ca3af", label="Perfect calibration")
    ax.plot(mp, fo, "o-", color=PRIMARY_COLOR, label="Proposed model")
    ax.set_ylabel("Observed methylation rate")
    ax.set_title(f"Calibration (cross-fitted Platt scaling)\nBrier {np.mean((prob - y) ** 2):.3f} | "
                 f"ECE {ece(y, prob):.3f}")
    ax.legend(fontsize=7, frameon=False)
    ax2.hist(prob, bins=20, color=PRIMARY_COLOR, alpha=0.7); ax2.set_xlabel("Predicted probability")
    ax2.set_ylabel("Patients"); save(fig, "fig8_calibration")


# ---------------------------------------------------------------- fig 9
def fig_decision_curve():
    prob = calibrated(); n = len(y); prev = y.mean(); th = np.linspace(0.05, 0.95, 91)
    nb_model, nb_all = [], []
    for t in th:
        pred = prob >= t; tp = ((pred == 1) & (y == 1)).sum(); fp = ((pred == 1) & (y == 0)).sum()
        nb_model.append(tp / n - fp / n * t / (1 - t)); nb_all.append(prev - (1 - prev) * t / (1 - t))
    fig, ax = plt.subplots(figsize=(5.2, 3.8))
    ax.plot(th, nb_model, color=PRIMARY_COLOR, lw=2, label="Proposed model")
    ax.plot(th, nb_all, color="#b45309", lw=1.2, label="Treat all as methylated")
    ax.axhline(0, color="k", lw=1, label="Treat none")
    ax.set_ylim(-0.05, max(max(nb_model), prev) + 0.05)
    ax.set_xlabel("Threshold probability"); ax.set_ylabel("Net benefit")
    ax.set_title("Decision curve analysis"); ax.legend(fontsize=7, frameon=False)
    better = th[np.array(nb_model) > np.maximum(np.array(nb_all), 0) + 1e-9]
    if len(better): note(f"Decision curve: model has higher net benefit than treat-all/none for thresholds "
                         f"{better.min():.2f}-{better.max():.2f}")
    save(fig, "fig9_decision_curve")


# ---------------------------------------------------------------- fig 10
def fig_selective():
    prob = calibrated(); conf = np.abs(prob - 0.5); order_ = np.argsort(-conf); pred = (prob >= 0.5).astype(int)
    cov = np.linspace(0.1, 1.0, 46); acc = []
    for c in cov:
        k = max(1, int(round(c * len(y)))); i = order_[:k]; acc.append((pred[i] == y[i]).mean())
    rows = []
    for c in (1.0, 0.5, 0.3, 0.2):
        k = int(round(c * len(y))); i = order_[:k]
        rows.append({"coverage": f"{int(c*100)}%", "patients": k, "accuracy": round(float((pred[i] == y[i]).mean()), 3)})
    pd.DataFrame(rows).to_csv(os.path.join(FIG, "table_selective.csv"), index=False)
    for r in rows: note(f"Selective prediction: coverage {r['coverage']} ({r['patients']} pts) accuracy {r['accuracy']}")
    fig, ax = plt.subplots(figsize=(5, 3.6))
    ax.plot(cov * 100, acc, color=PRIMARY_COLOR, lw=2)
    for r in rows[1:]:
        ax.plot(int(r["coverage"][:-1]), r["accuracy"], "o", color="#b91c1c")
        ax.annotate(f"{r['accuracy']:.2f}", (int(r["coverage"][:-1]), r["accuracy"]), textcoords="offset points",
                    xytext=(4, 4), fontsize=7)
    ax.set_xlabel("Coverage: % of patients with the most confident predictions"); ax.set_ylabel("Accuracy")
    ax.invert_xaxis(); ax.set_title("Confidence-based selective prediction")
    save(fig, "fig10_selective")


# ---------------------------------------------------------------- fig 11
def fig_ablation():
    f = os.path.join(OUT, "ablation_table.csv")
    if not os.path.isfile(f): print("  no ablation_table.csv -> fig11 skipped"); return
    t = pd.read_csv(f); t = t[t.dAUC.astype(str) != "-"].copy()
    t["d"] = t.dAUC.astype(float); t["p"] = t["DeLong p vs full"].astype(float)
    fig, ax = plt.subplots(figsize=(6.4, 0.45 * len(t) + 1))
    cols = ["#b91c1c" if p < 0.05 else "#9ca3af" for p in t.p]
    ax.barh(range(len(t)), t.d, color=cols)
    for k, (d, p) in enumerate(zip(t.d, t.p)):
        ax.text(d + (-0.002 if d < 0 else 0.002), k, f"{d:+.3f} (p={p:.3f})", va="center",
                ha="right" if d < 0 else "left", fontsize=7)
    ax.set_yticks(range(len(t))); ax.set_yticklabels(t.Variant, fontsize=8); ax.axvline(0, color="k", lw=1)
    lo = min(t.d.min(), 0) - 0.03; hi = max(t.d.max(), 0) + 0.03; ax.set_xlim(lo, hi)
    ax.set_xlabel("Change in AUC when the component is removed (red: DeLong p < 0.05)")
    ax.set_title("Ablation of the proposed model"); ax.invert_yaxis(); save(fig, "fig11_ablation")
    for _, r in t.iterrows(): note(f"Ablation {r.Variant}: dAUC {r.d:+.3f}, p {r.p:.4f}, strict dAUC {r['dAUC strict']}")


# ---------------------------------------------------------------- efficiency table
def table_efficiency():
    try:
        import torch.nn as nn, timm
    except Exception:
        print("  torch/timm not available -> efficiency table skipped"); return
    def count(name):
        if name == "simple_cnn":
            blk = lambda i, o: nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
            m = nn.Sequential(blk(3, 32), blk(32, 64), blk(64, 128), blk(128, 256), nn.AdaptiveAvgPool2d(1),
                              nn.Flatten(), nn.Linear(256, 1))
        else:
            m = timm.create_model(name, pretrained=False, num_classes=1)
        return sum(p.numel() for p in m.parameters()) / 1e6
    known = [("VGG-16", "vgg16_bn"), ("DenseNet-121", "densenet121"), ("ResNet-18", "resnet18"),
             ("MobileNet-v2", "mobilenetv2_100"), ("MobileNet-v3-Large", "mobilenetv3_large_100"),
             ("Simple CNN", "simple_cnn")]
    # only the baselines that are in this run's results table
    specs = [("Proposed (EfficientNet-B0 + B2)", ["efficientnet_b0", "efficientnet_b2"])] + \
            [(lab, [net]) for lab, net in known if any(n.startswith(f"Baseline {lab}") for n in baselines)]
    rows = []
    for lab, nets in specs:
        per = sum(count(n) for n in nets)
        rows.append({"Model": lab, "Networks per MRI type": len(nets), "Params per MRI type (M)": round(per, 2),
                     "Networks per patient (4 MRI types)": 4 * len(nets), "Total params per patient (M)": round(4 * per, 1)})
    t = pd.DataFrame(rows); t.to_csv(os.path.join(FIG, "table_efficiency.csv"), index=False)
    print(t.to_string(index=False))
    for r in rows: note(f"Params {r['Model']}: {r['Total params per patient (M)']} M total")


for name, fn in [("ROC/PR", fig_roc_pr), ("forest + Holm", fig_forest_and_holm), ("bootstrap", fig_bootstrap),
                 ("per-fold", fig_perfold), ("confusion + clinical", fig_confusion_and_clinical),
                 ("external", fig_external), ("calibration", fig_calibration),
                 ("decision curve", fig_decision_curve), ("selective", fig_selective),
                 ("ablation", fig_ablation), ("efficiency", table_efficiency)]:
    print(f"\n--- {name}")
    run(name, fn)

gc = [f for f in glob.glob(os.path.join(OUT, "fig_gradcam.*"))]
for f in gc:
    import shutil; shutil.copy(f, os.path.join(FIG, os.path.basename(f).replace("fig_gradcam", "fig12_gradcam")))
if gc: print("\n--- Grad-CAM figure copied as fig12_gradcam")

with open(os.path.join(FIG, "paper_numbers.txt"), "w") as fh:
    fh.write("\n".join(numbers) + "\n")
print(f"\nAll figures and tables are in {FIG}")
''',
    'check_inputs.py': r'''# =============================================================================
# check_inputs.py  --  run BEFORE kaggle_merged_training.py (takes ~1 minute)
# Checks that every Kaggle input is the correct dataset, plus GPUs and internet.
#   !python $(find /kaggle/input -name check_inputs.py | head -1)
# =============================================================================
import os, re, glob, subprocess, urllib.request
import pandas as pd

ROOT = os.environ.get("MERGED_INPUT_ROOT", "/kaggle/input")
MAX_FILES = 300_000           # stop walking huge inputs (e.g. a DICOM copy) early
results = []


def check(name, ok, detail, fix=""):
    results.append((name, ok, detail, fix))


def print_run_mode():
    """Say whether this is a background commit (safe) or an interactive session."""
    mode = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
    bar = "#" * 78
    if mode == "Batch":
        msg = ("RUN MODE: BACKGROUND COMMIT (Save & Run All)\n"
               "Safe to close the laptop. Output is kept under this version's Output tab.")
    elif mode == "Interactive":
        msg = ("RUN MODE: INTERACTIVE SESSION\n"
               "Output is LOST when the session stops. Use this for the quick test only;\n"
               "for the full run use Save Version -> Save & Run All (Commit).")
    else:
        msg = "RUN MODE: not on Kaggle (local run)"
    print(f"{bar}\n{msg}\n{bar}\n", flush=True)


print_run_mode()

# ---- one walk over all inputs ---------------------------------------------------
files = []
for top in sorted(glob.glob(os.path.join(ROOT, "*"))):
    n = 0
    for root, dirs, fs in os.walk(top, followlinks=True):
        dirs[:] = [d for d in dirs if d not in ("FLAIR", "T1w", "T1wCE", "T2w")]  # RSNA DICOM folders
        files += [os.path.join(root, f) for f in fs]; n += len(fs)
        if n > MAX_FILES: break
names = [os.path.basename(f) for f in files]
print(f"scanned {len(files):,} files under {ROOT}\n")


def find(pattern):
    rx = re.compile(pattern, re.I)
    return [f for f, b in zip(files, names) if rx.search(b)]


# ---- 1. GPUs --------------------------------------------------------------------
try:
    gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:
    gpus = []
check("GPU T4 x2", len(gpus) >= 2, f"{len(gpus)} GPU(s): " + "; ".join(g.split(" (")[0] for g in gpus),
      "Settings -> Accelerator -> GPU T4 x2")

# ---- 2. Internet (needed for ImageNet weights) ------------------------------------
try:
    urllib.request.urlopen("https://huggingface.co", timeout=10); net = True
except Exception:
    net = False
check("Internet", net, "on" if net else "off", "Settings -> Internet -> On (phone-verified account)")

# ---- 3. RSNA-MICCAI labels ----------------------------------------------------------
lab = find(r"^train_labels\.csv$")
if lab:
    d = pd.read_csv(lab[0]); ok = {"BraTS21ID", "MGMT_value"} <= set(d.columns) and len(d) >= 580
    check("RSNA-MICCAI competition", ok, f"train_labels.csv, {len(d)} patients", "add the competition input")
else:
    check("RSNA-MICCAI competition", False, "train_labels.csv not found",
          "add kaggle.com/competitions/rsna-miccai-brain-tumor-radiogenomic-classification")

# ---- 4. BraTS 2021 images -------------------------------------------------------------
tar = find(r"BraTS2021_Training_Data\.tar$")
segs = find(r"^BraTS2021_\d+_seg\.nii(\.gz)?$")
if tar:
    gb = os.path.getsize(tar[0]) / 1e9
    check("BraTS 2021 Task 1", gb > 5, f"tar found ({gb:.1f} GB)", "use dschettler8845/brats-2021-task1")
else:
    check("BraTS 2021 Task 1", len(segs) >= 1200, f"{len(segs)} patient folders with masks",
          "add kaggle.com/datasets/dschettler8845/brats-2021-task1")

# ---- 5. UPenn-GBM (NIfTI copy) ---------------------------------------------------------
up_img = find(r"^UPENN-GBM-\d+_11_FLAIR\.nii(\.gz)?$")
up_seg = find(r"^UPENN-GBM-\d+_11_.*segm\.nii(\.gz)?$")
up_dcm = [f for f in files if "UPENN-GBM" in f and f.lower().endswith(".dcm")]
if up_dcm and not up_img:
    check("UPenn-GBM", False, "WRONG COPY: DICOM files (.dcm) found, no NIfTI",
          "remove it; add kaggle.com/datasets/sanjanatg26/upenn-gbm")
else:
    subs = {re.match(r"(UPENN-GBM-\d+_11)", os.path.basename(f)).group(1) for f in up_img}
    check("UPenn-GBM (NIfTI)", len(subs) >= 550 and len(up_seg) >= 550,
          f"{len(subs)} patients with FLAIR, {len(up_seg)} tumour masks",
          "add kaggle.com/datasets/sanjanatg26/upenn-gbm")

# ---- 6. UCSF-PDGM --------------------------------------------------------------------
uc_seg = find(r"^UCSF-PDGM-\d+_tumor_segmentation\.nii(\.gz)?$")
uc_t1 = find(r"^UCSF-PDGM-\d+_T1\.nii(\.gz)?$")
uc_meta = find(r"UCSF-PDGM.*metadata.*\.csv$")
idx_ok = False
if uc_meta:
    m = pd.read_csv(uc_meta[0]); idx_ok = any("mgmt" in c.lower() and "index" in c.lower() for c in m.columns)
check("UCSF-PDGM", len(uc_seg) >= 450 and len(uc_t1) >= 450 and idx_ok,
      f"{len(uc_seg)} tumour masks, {len(uc_t1)} T1 scans, metadata "
      + ("with MGMT index" if idx_ok else "MISSING or no MGMT index"),
      "add kaggle.com/datasets/rukiyeaydn/ucsf-pdgm-filtered")

# ---- 7. your code dataset (mgmt-merged-code) -------------------------------------------
script = find(r"^kaggle_merged_training\.py$")
new_ver = bool(script) and 'SCRIPT_VERSION = "v9' in open(script[0]).read()
check("Training script", new_ver, "found, latest version" if new_ver else
      ("found but OLD version" if script else "not found"),
      "upload the 'Kaggle upload' folder as dataset mgmt-merged-code (New Version if it exists)")
figs = find(r"^paper_figures\.py$")
check("Figures script", bool(figs), "found" if figs else "not found",
      "upload the latest mgmt-merged-code.zip (it contains paper_figures.py)")
done = {os.path.basename(f) for f in files
        if f.endswith(".npz") and f.replace("\\", "/").split("/")[-3:-1] == ["outputs_merged", "jobs"]}
single = {d for d in done if d.startswith("ablsingle_")}
check("Proposed single-slice models (run 3)", len(single) >= 80,
      f"{len(single)} found (80 expected)",
      "add your dataset mgmt-merged-run3 (outputs_merged_run3.zip) as an input, "
      "otherwise the proposed model is retrained too (~1.5 h extra)")
ul = find(r"^upenn_mgmt_labels\.csv$")
check("UPenn MGMT labels", bool(ul) and len(pd.read_csv(ul[0])) == 291,
      f"{len(pd.read_csv(ul[0]))} labelled patients" if ul else "not found",
      "upload upenn_mgmt_labels.csv into mgmt-merged-code")
oof = find(r"^oof_predictions\.csv$")
check("Paper predictions (0.669)", bool(oof) and len(pd.read_csv(oof[0])) == 574,
      f"{len(pd.read_csv(oof[0]))} patients" if oof else "not found (comparison vs paper will be skipped)",
      "upload oof_predictions.csv into mgmt-merged-code")

# ---- report ---------------------------------------------------------------------------
w = max(len(r[0]) for r in results)
for name, ok, detail, fix in results:
    print(f"{'OK  ' if ok else 'FAIL'}  {name:<{w}}  {detail}" + ("" if ok else f"\n      -> fix: {fix}"))
bad = [r for r in results if not r[1]]
print("\n" + ("ALL GOOD - run kaggle_merged_training.py" if not bad
              else f"{len(bad)} problem(s) - fix them, then run this check again"))
''',
    'upenn_mgmt_labels.csv': r'''subject_id,MGMT_value
UPENN-GBM-00022_11,0
UPENN-GBM-00034_11,0
UPENN-GBM-00088_11,0
UPENN-GBM-00091_11,0
UPENN-GBM-00092_11,0
UPENN-GBM-00093_11,0
UPENN-GBM-00093_21,0
UPENN-GBM-00094_11,0
UPENN-GBM-00095_11,1
UPENN-GBM-00098_11,1
UPENN-GBM-00115_11,1
UPENN-GBM-00117_11,0
UPENN-GBM-00120_11,0
UPENN-GBM-00121_11,0
UPENN-GBM-00122_11,0
UPENN-GBM-00122_21,0
UPENN-GBM-00124_11,1
UPENN-GBM-00125_11,0
UPENN-GBM-00128_11,1
UPENN-GBM-00128_21,1
UPENN-GBM-00129_11,0
UPENN-GBM-00130_21,1
UPENN-GBM-00131_11,1
UPENN-GBM-00132_11,0
UPENN-GBM-00133_11,0
UPENN-GBM-00134_11,0
UPENN-GBM-00134_21,0
UPENN-GBM-00135_11,1
UPENN-GBM-00136_11,0
UPENN-GBM-00138_11,1
UPENN-GBM-00139_11,0
UPENN-GBM-00140_11,0
UPENN-GBM-00140_21,0
UPENN-GBM-00141_11,0
UPENN-GBM-00143_11,0
UPENN-GBM-00144_11,0
UPENN-GBM-00145_11,0
UPENN-GBM-00146_11,0
UPENN-GBM-00147_11,0
UPENN-GBM-00148_11,0
UPENN-GBM-00148_21,0
UPENN-GBM-00149_11,0
UPENN-GBM-00150_11,0
UPENN-GBM-00150_21,0
UPENN-GBM-00151_11,0
UPENN-GBM-00183_21,1
UPENN-GBM-00197_11,1
UPENN-GBM-00197_21,1
UPENN-GBM-00240_11,1
UPENN-GBM-00244_11,0
UPENN-GBM-00246_11,0
UPENN-GBM-00260_11,1
UPENN-GBM-00262_11,0
UPENN-GBM-00263_11,1
UPENN-GBM-00264_11,1
UPENN-GBM-00267_11,0
UPENN-GBM-00267_21,0
UPENN-GBM-00269_11,1
UPENN-GBM-00282_11,0
UPENN-GBM-00287_11,1
UPENN-GBM-00290_11,0
UPENN-GBM-00294_11,0
UPENN-GBM-00301_11,0
UPENN-GBM-00302_11,1
UPENN-GBM-00304_11,0
UPENN-GBM-00310_11,1
UPENN-GBM-00312_11,0
UPENN-GBM-00312_21,0
UPENN-GBM-00314_21,0
UPENN-GBM-00316_11,0
UPENN-GBM-00336_11,0
UPENN-GBM-00344_11,1
UPENN-GBM-00344_21,1
UPENN-GBM-00350_11,1
UPENN-GBM-00351_11,1
UPENN-GBM-00352_11,0
UPENN-GBM-00352_21,1
UPENN-GBM-00354_11,0
UPENN-GBM-00355_21,0
UPENN-GBM-00356_11,1
UPENN-GBM-00358_11,1
UPENN-GBM-00359_11,1
UPENN-GBM-00360_11,1
UPENN-GBM-00362_11,0
UPENN-GBM-00363_11,1
UPENN-GBM-00367_11,0
UPENN-GBM-00368_11,0
UPENN-GBM-00369_11,0
UPENN-GBM-00370_11,0
UPENN-GBM-00371_11,1
UPENN-GBM-00373_11,0
UPENN-GBM-00375_11,0
UPENN-GBM-00376_11,0
UPENN-GBM-00377_11,1
UPENN-GBM-00378_11,1
UPENN-GBM-00379_11,0
UPENN-GBM-00380_11,1
UPENN-GBM-00381_11,1
UPENN-GBM-00384_11,0
UPENN-GBM-00385_11,0
UPENN-GBM-00387_11,0
UPENN-GBM-00388_11,1
UPENN-GBM-00390_11,0
UPENN-GBM-00391_11,0
UPENN-GBM-00392_11,1
UPENN-GBM-00393_11,0
UPENN-GBM-00395_11,1
UPENN-GBM-00398_11,0
UPENN-GBM-00399_11,1
UPENN-GBM-00401_11,1
UPENN-GBM-00402_11,0
UPENN-GBM-00403_11,1
UPENN-GBM-00404_11,1
UPENN-GBM-00405_11,0
UPENN-GBM-00406_11,1
UPENN-GBM-00407_11,1
UPENN-GBM-00408_11,0
UPENN-GBM-00409_11,1
UPENN-GBM-00410_11,0
UPENN-GBM-00411_11,0
UPENN-GBM-00412_11,1
UPENN-GBM-00413_11,0
UPENN-GBM-00414_11,1
UPENN-GBM-00415_11,0
UPENN-GBM-00416_11,0
UPENN-GBM-00418_11,1
UPENN-GBM-00419_11,0
UPENN-GBM-00420_11,1
UPENN-GBM-00421_11,0
UPENN-GBM-00423_11,1
UPENN-GBM-00424_11,1
UPENN-GBM-00425_11,1
UPENN-GBM-00426_11,0
UPENN-GBM-00428_11,0
UPENN-GBM-00430_11,1
UPENN-GBM-00431_11,0
UPENN-GBM-00432_11,0
UPENN-GBM-00434_11,0
UPENN-GBM-00435_11,0
UPENN-GBM-00436_11,0
UPENN-GBM-00438_11,1
UPENN-GBM-00439_11,1
UPENN-GBM-00442_11,1
UPENN-GBM-00443_11,1
UPENN-GBM-00444_11,1
UPENN-GBM-00445_11,1
UPENN-GBM-00446_11,1
UPENN-GBM-00447_11,0
UPENN-GBM-00448_11,0
UPENN-GBM-00449_11,0
UPENN-GBM-00452_11,0
UPENN-GBM-00453_11,1
UPENN-GBM-00454_11,0
UPENN-GBM-00455_11,1
UPENN-GBM-00456_11,0
UPENN-GBM-00457_11,0
UPENN-GBM-00458_11,0
UPENN-GBM-00459_11,0
UPENN-GBM-00460_11,0
UPENN-GBM-00461_11,1
UPENN-GBM-00462_11,0
UPENN-GBM-00463_11,0
UPENN-GBM-00464_11,0
UPENN-GBM-00469_11,1
UPENN-GBM-00470_11,0
UPENN-GBM-00471_11,0
UPENN-GBM-00473_11,0
UPENN-GBM-00474_11,1
UPENN-GBM-00475_11,0
UPENN-GBM-00476_11,0
UPENN-GBM-00479_11,0
UPENN-GBM-00480_11,0
UPENN-GBM-00481_11,1
UPENN-GBM-00482_11,0
UPENN-GBM-00483_11,0
UPENN-GBM-00484_11,0
UPENN-GBM-00486_11,0
UPENN-GBM-00487_11,0
UPENN-GBM-00488_11,1
UPENN-GBM-00489_11,0
UPENN-GBM-00490_11,0
UPENN-GBM-00492_11,0
UPENN-GBM-00493_11,0
UPENN-GBM-00494_11,1
UPENN-GBM-00495_11,0
UPENN-GBM-00496_11,1
UPENN-GBM-00498_11,1
UPENN-GBM-00499_11,0
UPENN-GBM-00500_11,0
UPENN-GBM-00502_11,0
UPENN-GBM-00504_11,1
UPENN-GBM-00505_11,0
UPENN-GBM-00506_11,1
UPENN-GBM-00508_11,0
UPENN-GBM-00509_11,1
UPENN-GBM-00512_11,1
UPENN-GBM-00513_11,0
UPENN-GBM-00514_11,1
UPENN-GBM-00515_11,0
UPENN-GBM-00516_11,1
UPENN-GBM-00517_11,0
UPENN-GBM-00518_11,1
UPENN-GBM-00519_11,0
UPENN-GBM-00520_11,1
UPENN-GBM-00521_11,1
UPENN-GBM-00522_11,0
UPENN-GBM-00523_11,0
UPENN-GBM-00524_11,1
UPENN-GBM-00525_11,0
UPENN-GBM-00526_11,1
UPENN-GBM-00527_11,0
UPENN-GBM-00528_11,0
UPENN-GBM-00529_11,0
UPENN-GBM-00530_11,1
UPENN-GBM-00531_11,0
UPENN-GBM-00532_11,1
UPENN-GBM-00533_11,0
UPENN-GBM-00534_11,1
UPENN-GBM-00535_11,0
UPENN-GBM-00536_11,1
UPENN-GBM-00537_11,0
UPENN-GBM-00538_11,1
UPENN-GBM-00539_11,1
UPENN-GBM-00540_11,1
UPENN-GBM-00544_11,1
UPENN-GBM-00546_11,1
UPENN-GBM-00547_11,0
UPENN-GBM-00549_11,0
UPENN-GBM-00551_11,1
UPENN-GBM-00552_11,0
UPENN-GBM-00553_11,1
UPENN-GBM-00554_11,1
UPENN-GBM-00555_11,1
UPENN-GBM-00556_11,1
UPENN-GBM-00557_11,1
UPENN-GBM-00558_11,0
UPENN-GBM-00559_11,1
UPENN-GBM-00560_11,0
UPENN-GBM-00562_11,0
UPENN-GBM-00563_11,0
UPENN-GBM-00564_11,1
UPENN-GBM-00565_11,1
UPENN-GBM-00566_11,1
UPENN-GBM-00567_11,1
UPENN-GBM-00568_11,1
UPENN-GBM-00569_11,1
UPENN-GBM-00570_11,0
UPENN-GBM-00572_11,0
UPENN-GBM-00574_11,0
UPENN-GBM-00575_11,0
UPENN-GBM-00576_11,0
UPENN-GBM-00578_11,1
UPENN-GBM-00579_11,1
UPENN-GBM-00580_11,1
UPENN-GBM-00581_11,0
UPENN-GBM-00583_11,0
UPENN-GBM-00584_11,0
UPENN-GBM-00585_11,0
UPENN-GBM-00586_11,1
UPENN-GBM-00587_11,0
UPENN-GBM-00588_11,1
UPENN-GBM-00590_11,0
UPENN-GBM-00591_11,0
UPENN-GBM-00592_11,1
UPENN-GBM-00593_11,0
UPENN-GBM-00594_11,1
UPENN-GBM-00595_11,1
UPENN-GBM-00596_11,0
UPENN-GBM-00597_11,0
UPENN-GBM-00598_11,1
UPENN-GBM-00599_11,0
UPENN-GBM-00602_11,1
UPENN-GBM-00604_11,0
UPENN-GBM-00605_11,0
UPENN-GBM-00606_11,1
UPENN-GBM-00607_11,1
UPENN-GBM-00608_11,1
UPENN-GBM-00609_11,0
UPENN-GBM-00612_21,1
UPENN-GBM-00616_21,0
UPENN-GBM-00617_21,0
UPENN-GBM-00618_21,0
UPENN-GBM-00619_21,0
UPENN-GBM-00621_21,0
UPENN-GBM-00622_21,1
UPENN-GBM-00623_21,1
UPENN-GBM-00624_21,0
UPENN-GBM-00625_21,0
UPENN-GBM-00626_21,0
UPENN-GBM-00629_21,0
UPENN-GBM-00630_21,1
''',
    'rebuild_run4.py': r'''"""Rebuild MGMT Run 4 results from the archived model predictions."""
from __future__ import annotations

import argparse
import hashlib
import json
from contextlib import ExitStack
from io import BytesIO
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold

from kaggle_merged_training import auc_ci, delong, rankise, skill_weighted

MODALITIES = ("FLAIR", "T1w", "T1wCE", "T2w")
BACKBONES = ("b2", "b0")
ESTIMATES = ("checkpoint_selected", "fixed_epoch")
BASELINES = {
    "DenseNet-121": ("densenet121", "densenet_121"),
    "ResNet-18": ("resnet18", "resnet_18"),
    "MobileNet-v2": ("mobilenetv2_100", "mobilenet_v2"),
    "Four-layer CNN": ("simple_cnn", "simple_cnn_4_layer_from_scratch"),
}


def archive_members(archive):
    return {
        PurePosixPath(name).name: name
        for name in archive.namelist()
        if not name.endswith("/") and "__MACOSX" not in PurePosixPath(name).parts
    }


def open_run(stack, path):
    archive = stack.enter_context(ZipFile(path))
    members = archive_members(archive)
    if "worker_spec.json" in members:
        return archive, members
    nested = members.get("outputs_merged_run4.zip")
    if not nested:
        raise ValueError("Supply the Run 4 ZIP or its inner outputs_merged_run4.zip.")
    archive = stack.enter_context(ZipFile(BytesIO(archive.read(nested))))
    return archive, archive_members(archive)


def rank_average(*scores):
    return np.mean([rankise(score) for score in scores], axis=0)


def holm(p_values):
    order = np.argsort(p_values)
    adjusted = np.empty(len(order))
    cumulative = 0.0
    for rank, index in enumerate(order):
        cumulative = max(cumulative, min(1.0, (len(order) - rank) * p_values[index]))
        adjusted[index] = cumulative
    return adjusted


def calibrate(scores, labels, folds):
    probabilities = np.empty(len(labels))
    for fold in np.unique(folds):
        test = folds == fold
        model = LogisticRegression(C=1e6).fit(scores[~test, None], labels[~test])
        probabilities[test] = model.predict_proba(scores[test, None])[:, 1]
    return probabilities


def classification_metrics(labels, predictions):
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        "accuracy": (tp + tn) / len(labels),
        "sensitivity": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "PPV": tp / (tp + fp) if tp + fp else 0.0,
        "NPV": tn / (tn + fn) if tn + fn else 0.0,
        "F1": f1_score(labels, predictions),
        "MCC": matthews_corrcoef(labels, predictions),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    }


def target_specificity_threshold(labels, probabilities, target=0.90):
    fpr, _, thresholds = roc_curve(labels, probabilities)
    return thresholds[np.flatnonzero(1 - fpr >= target)[-1]]


class Run4:
    """Read the saved jobs and check each validation fold against its run configuration."""

    def __init__(self, archive, members):
        self.archive = archive
        self.members = members
        spec = json.loads(self.read("worker_spec.json"))
        self.config = spec["cfg"]
        self.labels = np.array(self.config["brats_y"], dtype=int)
        self.patient_ids = np.array(self.config["brats_ids"], dtype=int)
        self.seeds = self.config["seeds"]
        self.splits = {
            seed: [test for _, test in StratifiedKFold(5, shuffle=True, random_state=seed)
                   .split(self.patient_ids, self.labels)]
            for seed in self.seeds
        }
        self.folds = np.empty(len(self.labels), dtype=int)
        for fold, test in enumerate(self.splits[self.seeds[0]]):
            self.folds[test] = fold
        self.scheduled_jobs = len(spec["jobs"])
        missing = [job["name"] for job in spec["jobs"] if job["name"] + ".npz" not in members]
        if missing:
            raise ValueError(f"Missing {len(missing)} scheduled jobs, starting with {missing[0]}.")
        self.archived_jobs = sum(name.endswith(".npz") for name in members)

    def read(self, name):
        if name not in self.members:
            raise FileNotFoundError(f"{name} is missing from the Run 4 archive.")
        return self.archive.read(self.members[name])

    def csv(self, name):
        return pd.read_csv(BytesIO(self.read(name)), float_precision="round_trip")

    def job(self, prefix, backbone, modality, seed, fold, keys):
        name = f"{prefix}_{backbone}_{modality}_s{seed}_f{fold}_e20.npz"
        with np.load(BytesIO(self.read(name)), allow_pickle=False) as data:
            if not np.array_equal(data["va_idx"], self.splits[seed][fold]):
                raise ValueError(f"Validation patient order differs for {name}.")
            return [data[key] for key in keys]

    def internal(self, prefix, backbone):
        result = {estimate: {} for estimate in ESTIMATES}
        for modality in MODALITIES:
            repetitions = []
            for seed in self.seeds:
                scores = np.full((2, len(self.labels)), np.nan)
                for fold, test in enumerate(self.splits[seed]):
                    scores[:, test] = self.job(
                        prefix, backbone, modality, seed, fold, ("best_va", "last_va")
                    )
                if not np.isfinite(scores).all():
                    raise ValueError(f"Incomplete predictions for {prefix}/{backbone}/{modality}.")
                repetitions.append(scores)
            for estimate, score in zip(ESTIMATES, np.mean(repetitions, axis=0)):
                result[estimate][modality] = score
        return result

    def external(self, site):
        result = {estimate: {} for estimate in ESTIMATES}
        for backbone in BACKBONES:
            for modality in MODALITIES:
                predictions = [
                    self.job(f"ext{site}SS", backbone, modality, seed, fold, ("best_te", "last_te"))
                    for seed in self.seeds
                    for fold in range(5)
                ]
                for estimate, score in zip(ESTIMATES, np.mean(predictions, axis=0)):
                    result[estimate][backbone, modality] = score
        return result

    def check_internal_csv(self, name, score, column="pred"):
        saved = self.csv(name)
        if not all((
            np.array_equal(saved.BraTS21ID, self.patient_ids),
            np.array_equal(saved.MGMT_value, self.labels),
            np.array_equal(saved.fold, self.folds),
        )):
            raise ValueError(f"Patient IDs, labels, or folds differ in {name}.")
        self.check_scores(name, saved[column], score)

    @staticmethod
    def check_scores(name, archived, rebuilt):
        if len(archived) != len(rebuilt) or not np.allclose(archived, rebuilt, atol=1e-15, rtol=0):
            raise ValueError(f"Rebuilt predictions differ from {name}.")


def fuse_modalities(by_modality, labels):
    return {estimate: skill_weighted(by_modality[estimate], labels) for estimate in ESTIMATES}


def combine_backbones(by_backbone):
    return {
        estimate: rank_average(*(by_backbone[b][estimate] for b in BACKBONES))
        for estimate in ESTIMATES
    }


def compare(labels, reference, alternatives):
    rows = []
    for estimate in ESTIMATES:
        family = []
        for name, scores in alternatives.items():
            a, b = reference[name][estimate], scores[estimate]
            z, p_value = delong(labels, a, b)
            family.append({
                "comparison": name, "estimate": estimate,
                "reference_auc": roc_auc_score(labels, a), "comparison_auc": roc_auc_score(labels, b),
                "reference_minus_comparison_auc": roc_auc_score(labels, a) - roc_auc_score(labels, b),
                "DeLong_z": z, "DeLong_p": p_value,
            })
        for row, adjusted in zip(family, holm([row["DeLong_p"] for row in family])):
            row["Holm_p"] = adjusted
        rows.extend(family)
    return rows


def write_table(output, name, rows):
    pd.DataFrame(rows).to_csv(output / name, index=False)


def summarize_internal(run, output):
    y = run.labels
    modal, fused = {}, {}
    for b in BACKBONES:
        modal[b] = run.internal("ablsingle", b)
        fused[b] = fuse_modalities(modal[b], y)
        run.check_internal_csv(f"oof_merged_merged_{b}_brats_upenn_ucsf.csv", fused[b][ESTIMATES[0]])
    for name, (backbone, slug) in BASELINES.items():
        modal[name] = run.internal("baseSS", backbone)
        fused[name] = fuse_modalities(modal[name], y)
        run.check_internal_csv(f"oof_merged_baseline_{slug}_brats_upenn_ucsf.csv", fused[name][ESTIMATES[0]])
    for b in BACKBONES:
        name = f"BraTS only {b}"
        modal[name] = run.internal("controlSS", b)
        fused[name] = fuse_modalities(modal[name], y)
        run.check_internal_csv(f"oof_merged_control_{b}_brats_only_this_code.csv", fused[name][ESTIMATES[0]])

    full = fused["Proposed"] = combine_backbones(fused)
    fused["BraTS-only ensemble"] = combine_backbones({b: fused[f"BraTS only {b}"] for b in BACKBONES})
    modal["Proposed"] = {
        estimate: {m: rank_average(*(modal[b][estimate][m] for b in BACKBONES)) for m in MODALITIES}
        for estimate in ESTIMATES
    }
    run.check_internal_csv("oof_merged_merged_ens_b2_b0_primary.csv", full[ESTIMATES[0]])

    rows, fold_rows = [], []
    for name, estimates in fused.items():
        for estimate, score in estimates.items():
            sequences = {**modal[name][estimate], "Fused": score} if name in modal else {"Fused": score}
            for sequence, predictions in sequences.items():
                probabilities = calibrate(predictions, y, run.folds)
                rows.append({
                    "model": name, "sequence": sequence, "estimate": estimate,
                    "AUC": roc_auc_score(y, predictions), "AP": average_precision_score(y, predictions),
                    **classification_metrics(y, probabilities >= 0.5),
                })
            for fold in np.unique(run.folds):
                test = run.folds == fold
                fold_rows.append({
                    "model": name, "estimate": estimate, "fold": int(fold + 1),
                    "AUC": roc_auc_score(y[test], score[test]),
                })
    write_table(output, "internal_metrics.csv", rows)
    write_table(output, "fold_auc.csv", fold_rows)
    write_table(output, "baseline_tests.csv", compare(
        y, {name: full for name in BASELINES}, {name: fused[name] for name in BASELINES}
    ))
    write_table(output, "data_addition_tests.csv", compare(
        y, {"B0": fused["b0"], "B2": fused["b2"], "Ensemble": full},
        {"B0": fused["BraTS only b0"], "B2": fused["BraTS only b2"],
         "Ensemble": fused["BraTS-only ensemble"]},
    ))

    ablations = {
        "multi_cohort_data_brats_only": fused["BraTS-only ensemble"],
        "ensemble_efficientnet_b2_only": fused["b2"],
        "skill_weighting_plain_mean_of_modalities": {
            estimate: rank_average(*(np.mean(
                [rankise(modal[b][estimate][m]) for m in MODALITIES], axis=0
            ) for b in BACKBONES))
            for estimate in ESTIMATES
        },
        "multimodal_fusion_flair_only": {
            estimate: modal["Proposed"][estimate]["FLAIR"] for estimate in ESTIMATES
        },
    }
    for prefix, label in (
        ("ablrandinitSS", "imagenet_pretraining_random_init"),
        ("ablnocropSS", "tumour_focus_crop_whole_slice_tumour_slice"),
        ("ablcenterSS", "tumour_guidance_centre_slice_whole_image"),
    ):
        ablations[label] = combine_backbones({
            b: fuse_modalities(run.internal(prefix, b), y) for b in BACKBONES
        })
    ablation_rows = []
    for name, estimates in {**ablations, "proposed_full": full}.items():
        for estimate, column in zip(ESTIMATES, ("pred", "pred_strict")):
            score = estimates[estimate]
            run.check_internal_csv(f"oof_ablation_{name}.csv", score, column)
            ablation_rows.append({
                "variant": name, "estimate": estimate, "AUC": roc_auc_score(y, score),
                **classification_metrics(y, calibrate(score, y, run.folds) >= 0.5),
            })
    write_table(output, "ablation_metrics.csv", ablation_rows)
    write_table(output, "ablation_tests.csv", compare(y, {name: full for name in ablations}, ablations))

    earlier = {
        prefix: combine_backbones({b: fuse_modalities(run.internal(prefix, b), y) for b in BACKBONES})
        for prefix in ("cv", "control", "ablrandinit")
    }
    three_slice_rows = []
    for name, reference, alternative in (
        ("single_slice_vs_three_slices", full, earlier["cv"]),
        ("three_slices_vs_BraTS_only", earlier["cv"], earlier["control"]),
        ("three_slices_vs_random_initialization", earlier["cv"], earlier["ablrandinit"]),
    ):
        rows = compare(y, {name: reference}, {name: alternative})
        for row in rows:
            del row["Holm_p"]
        three_slice_rows.extend(rows)
    write_table(output, "three_slice_comparisons.csv", three_slice_rows)
    return full, earlier["cv"]


def summarize_external(run, output):
    dedup = run.csv("dedup_report.csv")
    cohort_rows, external_rows = [], []
    for site, site_config in run.config["sites"].items():
        records = dedup[dedup.site == site]
        retained = records[~records.excluded]
        labels = np.array(site_config["y"], dtype=int)
        borderline = retained.max_cosine.to_numpy() > 0.90
        if (list(retained.subject_id) != site_config["subject_id"]
                or not np.array_equal(retained.site_label, labels)):
            raise ValueError(f"Deduplication and job configuration disagree for {site}.")
        cohort_rows.append({
            "cohort": site, "screened": len(records), "ID_matches": records.brats21_id_match.sum(),
            "image_matches": records.image_match.sum(), "duplicates_excluded": records.excluded.sum(),
            "retained": len(retained), "retained_methylated": int(labels.sum()),
            "borderline_retained": int(borderline.sum()), "max_retained_similarity": retained.max_cosine.max(),
        })
        sequences = run.external(site)
        archived = run.csv(f"external_preds_{site}.csv")
        if list(archived.subject_id) != list(retained.subject_id) or not np.array_equal(archived.MGMT_value, labels):
            raise ValueError(f"Held-out patient IDs or labels differ for {site}.")
        for estimate, predictions in sequences.items():
            score = rank_average(*predictions.values())
            if estimate == ESTIMATES[0]:
                run.check_scores(site, archived.pred, score)
            for keep, scope in ((np.ones(len(labels), dtype=bool), "all_retained"),
                                (~borderline, "exclude_borderline")):
                auc, low, high = auc_ci(labels[keep], score[keep])
                row = {
                    "cohort": site, "estimate": estimate, "scope": scope, "patients": int(keep.sum()),
                    "AUC": auc, "AUC_CI_low": low, "AUC_CI_high": high,
                    "AP": average_precision_score(labels[keep], score[keep]),
                }
                for modality in MODALITIES:
                    sequence_score = rank_average(*(predictions[b, modality] for b in BACKBONES))
                    row[f"{modality}_AUC"] = roc_auc_score(labels[keep], sequence_score[keep])
                external_rows.append(row)
    write_table(output, "cohort_summary.csv", cohort_rows)
    write_table(output, "external_metrics.csv", external_rows)
    return cohort_rows


def summarize_clinical(run, scores, output):
    labels, folds = run.labels, run.folds
    probabilities = calibrate(scores, labels, folds)
    default = probabilities >= 0.5
    per_fold = np.empty(len(labels), dtype=bool)
    threshold_rows = []
    for fold in np.unique(folds):
        test = folds == fold
        threshold = target_specificity_threshold(labels[~test], probabilities[~test])
        per_fold[test] = probabilities[test] >= threshold
        threshold_rows.append({"fold": int(fold + 1), "threshold": threshold})
    pooled_threshold = target_specificity_threshold(labels, probabilities)
    pooled = probabilities >= pooled_threshold
    rows = [
        {"procedure": "default_probability_0.5", **classification_metrics(labels, default)},
        {"procedure": "manuscript_other_fold_thresholds", **classification_metrics(labels, per_fold)},
        {"procedure": "original_plot_script_pooled_threshold", **classification_metrics(labels, pooled)},
    ]
    write_table(output, "clinical_metrics.csv", rows)
    write_table(output, "high_specificity_thresholds.csv", threshold_rows)

    edges = np.quantile(probabilities, np.linspace(0, 1, 11))
    bins = np.clip(np.searchsorted(edges, probabilities, side="right") - 1, 0, 9)
    bin_rows = []
    for bin_id in np.unique(bins):
        selected = bins == bin_id
        bin_rows.append({
            "bin": int(bin_id + 1), "patients": int(selected.sum()),
            "mean_probability": probabilities[selected].mean(),
            "observed_methylation_rate": labels[selected].mean(),
        })
    write_table(output, "calibration_bins.csv", bin_rows)
    brier = float(np.mean((probabilities - labels) ** 2))
    ece = float(sum(abs(row["mean_probability"] - row["observed_methylation_rate"])
                    * row["patients"] / len(labels) for row in bin_rows))
    (output / "calibration_summary.json").write_text(json.dumps({"Brier": brier, "ECE": ece}, indent=2))

    decisions = []
    for threshold in np.linspace(0.05, 0.95, 91):
        pos = probabilities >= threshold
        tp, fp = np.sum(pos & (labels == 1)), np.sum(pos & (labels == 0))
        odds = threshold / (1 - threshold)
        decisions.append({
            "threshold": threshold, "model": (tp - fp * odds) / len(labels),
            "all_methylated": labels.mean() - (1 - labels.mean()) * odds, "none_methylated": 0.0,
        })
    write_table(output, "decision_curve.csv", decisions)

    confidence = np.abs(probabilities - 0.5)
    ordered = np.argsort(-confidence)
    correct = default == labels
    coverages = (1.0, 0.5, 0.3, 0.2)
    sample_sizes = [round(len(labels) * coverage) for coverage in coverages]
    rng = np.random.default_rng(42)
    boot = np.empty((2000, len(coverages)))
    for i in range(len(boot)):
        sample = rng.integers(0, len(labels), len(labels))
        ranked_sample = sample[np.argsort(-confidence[sample])]
        boot[i] = [correct[ranked_sample[:n]].mean() for n in sample_sizes]
    write_table(output, "selective_prediction.csv", [
        {
            "coverage": coverage, "patients": n, "accuracy": correct[ordered[:n]].mean(),
            "recomputed_bootstrap_CI_low": np.quantile(boot[:, i], 0.025),
            "recomputed_bootstrap_CI_high": np.quantile(boot[:, i], 0.975),
        }
        for i, (coverage, n) in enumerate(zip(coverages, sample_sizes))
    ])
    write_table(output, "calibrated_predictions.csv", {
        "BraTS21ID": run.patient_ids, "fold": folds, "MGMT_value": labels,
        "fused_score": scores, "calibrated_probability": probabilities,
        "default_prediction": default.astype(int), "other_fold_high_specificity_prediction": per_fold.astype(int),
    })
    return rows[1]["specificity"]


def main(archive_path, output):
    output.mkdir(parents=True, exist_ok=True)
    with archive_path.open("rb") as archive_file:
        archive_digest = hashlib.file_digest(archive_file, "sha256").hexdigest()
    with ExitStack() as stack:
        run = Run4(*open_run(stack, archive_path))
        full, earlier = summarize_internal(run, output)
        cohorts = summarize_external(run, output)
        high_spec = summarize_clinical(run, full[ESTIMATES[0]], output)
        auc, low, high = auc_ci(run.labels, full[ESTIMATES[0]])
        summary = {
            "archive": archive_path.name,
            "archive_sha256": archive_digest,
            "patients": len(run.labels), "methylated": int(run.labels.sum()),
            "scheduled_jobs_present": run.scheduled_jobs, "archived_jobs": run.archived_jobs,
            "archived_predictions_reconstructed": True,
            "checkpoint_selected_AUC": auc, "checkpoint_selected_AUC_CI": [low, high],
            "fixed_epoch_AUC": roc_auc_score(run.labels, full[ESTIMATES[1]]),
            "three_slice_AUC": {estimate: roc_auc_score(run.labels, score) for estimate, score in earlier.items()},
            "duplicate_exclusions": int(sum(row["duplicates_excluded"] for row in cohorts)),
            "other_fold_high_specificity": high_spec,
        }
    (output / "verification_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    print(f"Checkpoint-selected AUC: {100 * auc:.2f}% (95% CI {100 * low:.2f}–{100 * high:.2f}%)")
    print(f"Fixed-epoch AUC: {100 * summary['fixed_epoch_AUC']:.2f}%")
    print(f"Duplicate exclusions: {summary['duplicate_exclusions']}")
    print(f"Other-fold specificity: {100 * high_spec:.2f}%")
    print(f"Saved results to {output.resolve()}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("archive", type=Path, help="Run 4 ZIP or outputs_merged_run4.zip")
    parser.add_argument("--output", type=Path, default=Path("mgmt_run4_results"))
    options = parser.parse_args()
    main(options.archive, options.output)
''',
}

for name, content in RECOVERED_FILES.items():
    (SOURCE_DIR / name).write_text(content, encoding="utf-8")

print(f"Source files are available in {SOURCE_DIR}")

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

def locate_run4():
    if RUN4_ARCHIVE.strip():
        selected = Path(RUN4_ARCHIVE).expanduser().resolve()
        if not selected.is_file():
            raise FileNotFoundError(f"Run 4 archive does not exist: {selected}")
        return selected

    kaggle_inputs = Path("/kaggle/input")
    files = list(kaggle_inputs.rglob("*.zip")) if kaggle_inputs.is_dir() else list(Path.cwd().glob("*.zip"))
    for pattern in (
        lambda p: p.name.lower() == "outputs_merged_run4.zip",
        lambda p: p.name.lower().startswith(("run 4", "run_4", "run-4")),
    ):
        matches = sorted(p.resolve() for p in files if pattern(p))
        if len(matches) > 1:
            options = "\n".join(str(p) for p in matches)
            raise ValueError(f"Multiple Run 4 archives found. Set RUN4_ARCHIVE to one of:\n{options}")
        if matches:
            return matches[0]

    base = kaggle_inputs if kaggle_inputs.is_dir() else Path.cwd()
    candidates = sorted({
        spec.parent.parent for spec in base.rglob("worker_spec.json")
        if spec.parent.name == "jobs" and (spec.parent.parent / "oof_ablation_proposed_full.csv").is_file()
    })
    if len(candidates) == 1:
        run = candidates[0]
        repacked = WORK_DIR / "outputs_merged_run4_detected.zip"
        with ZipFile(repacked, "w", compression=ZIP_DEFLATED) as target:
            for file in sorted(run.rglob("*")):
                if file.is_file() and file.suffix in {".npz", ".json", ".csv"}:
                    target.write(file, Path("outputs_merged") / file.relative_to(run))
        return repacked
    raise FileNotFoundError("Attach the Run 4 ZIP using Add Input or set RUN4_ARCHIVE to its path.")

run4 = locate_run4()
print(f"Using {run4}")
subprocess.run(
    [sys.executable, str(SOURCE_DIR / "rebuild_run4.py"), str(run4), "--output", str(OUTPUT_DIR)],
    check=True,
)

In [ ]:
import pandas as pd
from IPython.display import display, FileLink
from zipfile import ZIP_DEFLATED, ZipFile

metrics = pd.read_csv(OUTPUT_DIR / "internal_metrics.csv")
main = metrics.loc[
    (metrics["model"] == "Proposed") & (metrics["sequence"] == "Fused"),
    ["estimate", "AUC", "AP", "accuracy", "MCC"],
]
display(main.round(4))
display(pd.read_csv(OUTPUT_DIR / "clinical_metrics.csv").round(4))

download = WORK_DIR / "MGMT_Run4_rebuilt_results.zip"
with ZipFile(download, "w", compression=ZIP_DEFLATED) as archive:
    for file in sorted(OUTPUT_DIR.iterdir()):
        if file.is_file():
            archive.write(file, Path("results") / file.name)
    for name in RECOVERED_FILES:
        archive.write(SOURCE_DIR / name, Path("source") / name)
display(FileLink(str(download)))

## How to read the outputs

- `verification_summary.json` records the archive checksum and verified headline AUCs.
- `internal_metrics.csv`, `baseline_tests.csv`, `data_addition_tests.csv`, `ablation_metrics.csv`, and `three_slice_comparisons.csv` report the model comparisons. AUC and rates are proportions.
- `external_metrics.csv` includes both held-out cohorts with and without the borderline-similarity cases. Very small differences in external AUCs can occur when numerically near-tied ranks are parsed differently.
- `clinical_metrics.csv` reports both the other-fold high-specificity procedure used for the manuscript and the pooled procedure used by the original plotting script.
- `selective_prediction.csv` includes newly recomputed intervals. Each of 2,000 bootstrap resamples is ranked by confidence before coverage is applied. Some interval endpoints may differ from the manuscript.

## Optional fresh training

The recovered `kaggle_merged_training.py` is the v9 single-slice training script from the earlier source ZIP. To run it from scratch, use a separate Kaggle notebook with **T4 x2**, the RSNA-MICCAI labels, BraTS 2021, UPenn-GBM with its MGMT labels, and UCSF-PDGM with its metadata. Add the source package as an input and enable Internet for pretrained ImageNet weights. Run the recovered training script as a separate Python process. The output folder defaults to `/kaggle/working/outputs_merged`. Fresh training requires the original datasets and may differ numerically from the saved Run 4.